# LeetCode 297 — Serialize and Deserialize Binary Tree bằng Python (Hard)

Notebook này giảng nhập môn một bài **thiết kế class** (design) về cây nhị phân. Khung code Python3 mà LeetCode cho sẵn:

```python
# Definition for a binary tree node.
# class TreeNode(object):
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Codec:

    def serialize(self, root):
        """Encodes a tree to a single string.

        :type root: TreeNode
        :rtype: str
        """


    def deserialize(self, data):
        """Decodes your encoded data to tree.

        :type data: str
        :rtype: TreeNode
        """


# Your Codec object will be instantiated and called as such:
# ser = Codec()
# deser = Codec()
# ans = deser.deserialize(ser.serialize(root))
```

Mục tiêu của bài học:

1. Hiểu **cây nhị phân** (binary tree) từ con số 0: `TreeNode`, root, con trái/phải, lá; tự dựng cây từ list kiểu LeetCode và **in cây dạng ASCII** để nhìn thấy nó.
2. Hiểu **serialize / deserialize** là gì và **vì sao cần** (lưu file, gửi qua mạng) — vì sao không thể "lưu thẳng" một object.
3. Tự khám phá vì sao "ghi các giá trị theo pre-order" là **không đủ** (nhiều cây khác nhau cho cùng một chuỗi — có demo), và vì sao chỉ cần thêm ký hiệu `#` cho chỗ trống là đủ.
4. Nắm các công cụ Python cần cho bài: đệ quy, call stack và giới hạn đệ quy, `deque`, `str.join` / `split`, và đặc biệt cặp **`iter` + `next`**.
5. Đi qua các cách giải theo thứ tự tự nhiên: cách **"lười"** (JSON lồng nhau / `repr`) → **DFS pre-order có `#`** (cách chính, trace kỹ) → **BFS level-order** đúng định dạng LeetCode `[1,2,3,null,null,4,5]` → **post-order đọc ngược** và **pre-order + in-order** (tùy chọn) → **bản lặp dùng stack** cho cây rất sâu.
6. Chứng minh vì sao cách chính đúng, so sánh **độ dài chuỗi** giữa các cách (và việc cắt `null` thừa ở cuối), test round-trip trên cây ngẫu nhiên có `seed` lẫn cây lệch sâu `10^4` tầng, tránh các lỗi kinh điển và nhận dạng pattern cho các bài anh em (331, 449, 652...).

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> **Serialization** (tuần tự hóa) là quá trình chuyển một cấu trúc dữ liệu hay một object thành một dãy bit, để có thể lưu nó vào file hoặc bộ nhớ đệm, hoặc truyền nó qua một kết nối mạng, rồi sau đó **dựng lại** (reconstruct) ở cùng máy tính đó hoặc ở một máy tính khác.
>
> Hãy thiết kế một thuật toán để serialize và deserialize một **cây nhị phân**. Không có giới hạn nào về cách thuật toán serialize/deserialize của bạn hoạt động. Bạn chỉ cần bảo đảm rằng một cây nhị phân có thể được serialize thành **một chuỗi**, và chuỗi này có thể được deserialize trở lại **đúng cấu trúc cây ban đầu**.
>
> **Làm rõ:** định dạng input/output giống với cách LeetCode serialize một cây nhị phân. Bạn **không bắt buộc** phải theo định dạng này, hãy sáng tạo và tự nghĩ ra những cách tiếp cận khác nhau.

**Example 1**

```text
Input: root = [1,2,3,null,null,4,5]
Output: [1,2,3,null,null,4,5]
```

**Example 2**

```text
Input: root = []
Output: []
```

**Constraints**

- Số node trong cây nằm trong khoảng `[0, 10^4]`.
- `-1000 <= Node.val <= 1000`

### Ký hiệu dùng suốt notebook

- `n` = số node của cây (`0 <= n <= 10^4`, nghĩa là cây **có thể rỗng**).
- `h` = chiều cao của cây (số node trên đường dài nhất từ root xuống một lá). Cây cân bằng thì `h ≈ log₂ n` (với `n = 10^4` chỉ khoảng 14); cây "lệch" thành một đường thẳng thì `h = n`.

### 1.1 Đề thực sự kiểm tra điều gì?

Đọc kỹ 3 dòng comment cuối khung code: LeetCode tạo **hai object khác nhau** `ser` và `deser`, gọi `ser.serialize(root)` để được một chuỗi, đưa chuỗi đó cho `deser.deserialize(...)`, rồi lấy cây kết quả `ans` **in ra theo định dạng list** và so với input. Vì vậy ở Example 1, Output trùng với Input: nghĩa là "cây bạn dựng lại phải giống hệt cây ban đầu".

| Chi tiết của đề | Hệ quả khi thiết kế |
|---|---|
| `ser` và `deser` là **hai object khác nhau** | không thể "giấu" cây trong `self` rồi trả lại; mọi thông tin phải nằm **trong chuỗi** |
| định dạng chuỗi do ta **tự chọn** | được sáng tạo — miễn là `deserialize(serialize(T))` cho lại đúng cây `T` |
| `n` có thể bằng `0` | `serialize(None)` phải ra một chuỗi hợp lệ, và deserialize chuỗi đó phải ra `None` |
| `n` tới `10^4` | cây lệch có thể sâu `10^4` tầng, lớn hơn giới hạn đệ quy mặc định của Python (`1000`) |
| `-1000 <= val <= 1000` | có số **âm** (dấu `-`), số **0**, số nhiều chữ số; và giá trị có thể **trùng nhau** (chỉ có 2001 giá trị khác nhau mà cây tới `10^4` node, nên cây lớn chắc chắn có giá trị trùng) |
| `TreeNode(x)` chỉ nhận **1** tham số | phải tạo node trước rồi gán `.left` / `.right` sau |

Vài dòng cuối bảng nghe có vẻ vụn vặt, nhưng mỗi dòng sẽ ứng với một lỗi kinh điển ở mục 15.

## 2. Nền tảng: cây nhị phân từ căn bản

### 2.1 Từ vựng

**Cây nhị phân** (binary tree) là cấu trúc gồm các **node** (nút). Mỗi node có ba thứ:

- `val`: giá trị lưu ở node.
- `left`: con **trái** — một node khác, hoặc `None` nếu không có.
- `right`: con **phải** — một node khác, hoặc `None`.

Cây của Example 1, `[1,2,3,null,null,4,5]`, trông như sau:

```text
      1
     / \
    2   3
       / \
      4   5
```

| Thuật ngữ | Nghĩa | Ví dụ trên cây Example 1 |
|---|---|---|
| **root** (gốc) | node trên cùng, không có cha | `1` |
| **child** (con) / **parent** (cha) | node ngay dưới / ngay trên | `4`, `5` là con của `3`; `3` là cha của `4` |
| **leaf** (lá) | node không có con nào | `2`, `4`, `5` |
| **subtree** (cây con) | một node cùng toàn bộ con cháu của nó | cây con gốc `3` gồm `{3, 4, 5}` |
| **depth** (độ sâu) | số cạnh từ root đi xuống tới node | depth(`1`) = 0, depth(`3`) = 1, depth(`4`) = 2 |
| **height** (chiều cao) | số node trên đường dài nhất từ root xuống lá | 3 (đường `1 → 3 → 4`) |

Khác với `list` (truy cập bằng chỉ số `a[i]`), với cây ta chỉ cầm **node root** trong tay và đi xuống bằng `.left` / `.right`.

**Chú ý riêng cho bài 297:** LeetCode dùng định nghĩa `TreeNode` "kiểu cũ" — constructor chỉ nhận **một** tham số `x`, còn `left` và `right` luôn khởi đầu bằng `None`, ta tự gán sau. (Nhiều bài cây khác dùng `TreeNode(val=0, left=None, right=None)`; ở bài này mà gọi `TreeNode(1, None, None)` sẽ bị `TypeError`.) Ta định nghĩa lại **y hệt** để chạy trong notebook.

In [1]:
from collections import deque
import ast
import json
import os
import random
import sys
import tempfile
import time


class TreeNode(object):
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None


# Dựng TAY cây Example 1: tạo node trước, nối con sau.
root1 = TreeNode(1)
root1.left = TreeNode(2)
root1.right = TreeNode(3)
root1.right.left = TreeNode(4)
root1.right.right = TreeNode(5)

print(root1.val, root1.left.val, root1.right.val)   # 1 2 3
print(root1.right.left.val)                         # 4: đi phải (tới 3) rồi đi trái (tới 4)
print(root1.left.left)                              # None: node 2 không có con, nó là lá

1 2 3
4
None


### 2.2 Serialize / deserialize là gì? Vì sao cần?

Trong bộ nhớ (RAM), cây là một đám object nằm rải rác. Node `1` "biết" node `2` nhờ một **tham chiếu** (reference) — hãy hình dung đó là **địa chỉ ô nhớ** nơi node `2` đang nằm. Toàn bộ "hình dạng" của cây chỉ là thông tin *ai trỏ tới ai*.

Vấn đề là địa chỉ ô nhớ chỉ có nghĩa **bên trong chương trình đang chạy**:

- **Lưu file**: tắt chương trình thì RAM được giải phóng; ngày mai mở lại, địa chỉ cũ không còn trỏ tới đâu cả.
- **Gửi qua mạng**: máy bên kia có bộ nhớ riêng; địa chỉ `0x7f...` trên máy bạn hoàn toàn vô nghĩa với nó.
- File và đường truyền mạng chỉ chứa được **dãy byte** — trong thực tế thường là một **chuỗi ký tự**.

Vì vậy ta cần hai thao tác ngược nhau:

| Thao tác | Chiều | Ví dụ đời thường |
|---|---|---|
| **serialize** (tuần tự hóa, "mã hóa") | cây trong RAM → **chuỗi** | tháo cái tủ thành các tấm gỗ phẳng + một tờ hướng dẫn lắp, để gửi bưu điện |
| **deserialize** (giải tuần tự, "giải mã") | chuỗi → cây **mới** | người nhận đọc tờ hướng dẫn, lắp lại một cái tủ y hệt |

Cây dựng lại gồm các object **mới** (địa chỉ khác hẳn), nhưng **cùng hình dạng và cùng giá trị** — đó chính xác là điều đề yêu cầu. Bạn gặp serialize hằng ngày: JSON trong các web API, file lưu game, cache, dữ liệu gửi giữa các máy chủ...

Cell dưới cho thấy ba điều: in một object chỉ ra địa chỉ; file **không nhận** object; chỉ chuỗi mới lưu và gửi được. Ghi chú cú pháp:

- `tempfile.TemporaryDirectory()` tạo một thư mục tạm, tự xóa khi xong việc (để notebook không xả rác ra máy bạn).
- `with open(path, "w") as f:` mở file để **ghi** (`"w"` = write), tự đóng file khi ra khỏi khối `with`; `open(path)` không ghi gì thêm là mở để **đọc**.
- `.encode("utf-8")` đổi chuỗi thành `bytes` — dạng dữ liệu thật sự chạy trên mạng (chữ `b'...'` ở đầu cho biết đó là bytes).

In [2]:
print(root1)                         # chỉ là "địa chỉ" của object, vô nghĩa ngoài chương trình này
print("id(root1) =", id(root1))      # con số này đổi mỗi lần chạy lại notebook

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "tree.txt")
    with open(path, "w") as f:
        try:
            f.write(root1)                   # thử ghi thẳng object vào file
        except TypeError as e:
            print("Không ghi thẳng object được ->", e)
        f.write("1,2,3,null,null,4,5")       # tạm viết TAY một chuỗi mô tả cây
    with open(path) as f:
        loaded = f.read()

print("đọc lại từ file:", loaded, "| kiểu:", type(loaded).__name__)
print("dạng gửi qua mạng:", loaded.encode("utf-8"))

id(root1) = 140003774865696
Không ghi thẳng object được -> write() argument must be str, not TreeNode
đọc lại từ file: 1,2,3,null,null,4,5 | kiểu: str
dạng gửi qua mạng: b'1,2,3,null,null,4,5'


Chuỗi `"1,2,3,null,null,4,5"` ở trên do ta **viết tay**. Toàn bộ bài toán là: viết hàm **tự động** sinh ra một chuỗi như vậy từ cây (serialize), và hàm biến chuỗi đó trở lại thành cây (deserialize).

### 2.3 LeetCode viết cây thành list theo tầng (level-order)

Thật ra `[1,2,3,null,null,4,5]` **chính là một cách serialize** mà LeetCode dùng để in cây — ta sẽ tự cài lại nó ở mục 9. Cách đọc: đi **từng tầng, từ trái sang phải**:

```text
tầng 0:  1
tầng 1:  2, 3                  (con trái, con phải của 1)
tầng 2:  null, null, 4, 5      (con của 2 là null, null; con của 3 là 4, 5)
```

Quy tắc: lấy lần lượt từng node **thật** (không phải `null`) theo thứ tự tầng; mỗi node "tiêu thụ" **2 phần tử kế tiếp** của list làm con trái và con phải. `null` không có con nên không tiêu thụ thêm gì. Các `null` ở **cuối** list thường bị lược bỏ (con của 4 và 5 không được ghi).

Để làm đúng thứ tự "node nào gặp trước thì được gán con trước", ta dùng **hàng đợi** (queue): vào trước, ra trước (FIFO — First In, First Out), giống hàng người xếp mua vé. Python có sẵn `collections.deque`:

- `q.append(x)`: thêm `x` vào **cuối** hàng.
- `q.popleft()`: lấy ra phần tử ở **đầu** hàng — nhanh `O(1)`. (Còn `list.pop(0)` phải dời tất cả phần tử còn lại lên một ô, tốn `O(n)` — ta sẽ đo sự khác biệt này ở mục 15.)

In [3]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("hàng đợi:", list(q))
print("ra trước:", q.popleft())   # A vào trước nên ra trước
print("ra tiếp :", q.popleft())   # B
print("còn lại :", list(q))       # ['C']

hàng đợi: ['A', 'B', 'C']
ra trước: A
ra tiếp : B
còn lại : ['C']


### 2.4 Helper: `build_tree` (list → cây) và `tree_to_list` (cây → list)

Hai hàm tiện ích để tạo cây test và kiểm tra kết quả, viết **bằng vòng lặp** (không đệ quy) nên dùng được cả với cây rất sâu:

- `build_tree(values)`: tạo root từ `values[0]`, đưa vào queue. Mỗi lần `popleft` một node thì đọc **2 phần tử tiếp theo** làm con trái / con phải (bỏ qua `None`); con thật được xếp vào cuối queue để tới lượt nó nhận con. Chú ý ta kiểm tra `values[i] is not None` chứ **không** viết `if values[i]:` — vì giá trị `0` cũng là "sai" (falsy) trong Python và sẽ bị bỏ mất (lỗi kinh điển, mục 15).
- `tree_to_list(root)`: làm ngược lại — duyệt theo tầng, ghi `None` cho chỗ trống, rồi cắt các `None` thừa ở cuối. Đây là cách LeetCode in `Output`.

Nếu `build_tree` rồi `tree_to_list` cho lại đúng list ban đầu, ta tin được hai helper.

### 2.5 Helper: in cây dạng ASCII để **nhìn thấy** cây

Đọc list `[1,2,3,null,null,4,5]` rất khó hình dung. Hàm `print_tree` in cây theo kiểu cây thư mục: mỗi node một dòng, `L:` là con trái, `R:` là con phải, `·` nghĩa là "không có con ở phía này" (chỉ in khi phía kia có con, để thấy rõ con còn lại là trái hay phải — điều cực kỳ quan trọng trong bài này).

Hàm này tự gọi lại chính nó cho từng con — đó chính là **đệ quy**, sẽ học kỹ ở mục 3. Bây giờ chỉ cần dùng (và chỉ dùng cho cây nhỏ). Cell code thứ nhất dưới đây là `build_tree` / `tree_to_list`, cell thứ hai là `print_tree`.

In [4]:
def build_tree(values):
    # Dựng cây từ list level-order kiểu LeetCode (None = không có node).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:      # con trái
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:      # con phải
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    # Cây -> list level-order (đúng cách LeetCode in Output).
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:      # bỏ None thừa ở cuối
        out.pop()
    return out


ex1 = [1, 2, 3, None, None, 4, 5]
print(tree_to_list(build_tree(ex1)))
print(tree_to_list(root1))              # cây dựng tay ở trên cũng ra đúng list này
print(tree_to_list(build_tree([])))     # Example 2: cây rỗng
for vals in [ex1, [], [0, 0, 0], [1, None, 2, None, 3], [5, 4, 7, 3, None, 2, None, -1, None, 9]]:
    assert tree_to_list(build_tree(vals)) == vals
print("build_tree / tree_to_list OK")

[1, 2, 3, None, None, 4, 5]
[1, 2, 3, None, None, 4, 5]
[]
build_tree / tree_to_list OK


In [5]:
def print_tree(root):
    if root is None:
        print("(cây rỗng)")
        return
    print(root.val)

    def walk(node, prefix):
        if node.left is None and node.right is None:
            return
        for tag, child, last in (("L", node.left, False), ("R", node.right, True)):
            branch = "└── " if last else "├── "
            if child is None:
                print(prefix + branch + tag + ": ·")
            else:
                print(prefix + branch + f"{tag}: {child.val}")
                walk(child, prefix + ("    " if last else "│   "))

    walk(root, "")


print("Example 1:")
print_tree(root1)
print("\nExample 2:")
print_tree(build_tree([]))
print("\nCây lệch phải [1,null,2,null,3]:")
print_tree(build_tree([1, None, 2, None, 3]))

Example 1:
1
├── L: 2
└── R: 3
    ├── L: 4
    └── R: 5

Example 2:
(cây rỗng)

Cây lệch phải [1,null,2,null,3]:
1
├── L: ·
└── R: 2
    ├── L: ·
    └── R: 3


## 3. Nền tảng: đệ quy và các kiểu duyệt cây

### 3.1 Đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Cây rất hợp với đệ quy vì: *cây = root + cây con trái + cây con phải*, mà hai cây con cũng là cây.

Một hàm đệ quy luôn có 2 phần:

1. **Base case** (trường hợp cơ sở): bài toán nhỏ tới mức trả lời ngay. Với cây, gần như luôn là `if node is None: return ...`.
2. **Recursive case**: gọi hàm cho `node.left`, `node.right`, rồi **kết hợp** hai kết quả với `node.val`.

Ví dụ đếm số node và tính chiều cao:

```text
count(None) = 0                               height(None) = 0
count(node) = 1 + count(left) + count(right)  height(node) = 1 + max(height(left), height(right))
```

Với Example 1: `count(3) = 1 + 1 + 1 = 3`, `count(2) = 1`, nên `count(1) = 1 + 1 + 3 = 5`. Tương tự `height(3) = 2`, `height(1) = 1 + max(1, 2) = 3`.

### 3.2 Call stack và giới hạn đệ quy

Khi `count(1)` gọi `count(2)`, nó **tạm dừng** và chờ. Các lời gọi đang chờ được xếp chồng lên nhau trong **call stack** (ngăn xếp lời gọi): lời gọi nào vào sau thì xong trước. Độ cao của chồng này bằng **độ sâu** của node đang xét, tối đa là `h`.

Python giới hạn chồng này (mặc định khoảng `1000` tầng) để chương trình không ngốn hết bộ nhớ; vượt quá thì báo `RecursionError`. Cây cân bằng `10^4` node chỉ sâu khoảng 14 tầng — chẳng sao cả. Nhưng cây **lệch** `10^4` node thì sâu `10^4` tầng, và một hàm đệ quy "ngây thơ" sẽ vỡ. Ta sẽ gặp lại chuyện này ở mục 3.4, 4.3 và 11.

### 3.3 Ba kiểu duyệt DFS: pre-order, in-order, post-order

**DFS** (Depth-First Search — duyệt theo chiều sâu): đi xuống sâu hết một nhánh rồi mới quay lại nhánh khác. Cùng một khung đệ quy, chỉ cần đổi **vị trí của dòng "xử lý node"** là ra ba thứ tự:

| Tên | Thứ tự | Example 1 |
|---|---|---|
| **pre-order** (tiền thứ tự) | node → trái → phải | `[1, 2, 3, 4, 5]` |
| **in-order** (trung thứ tự) | trái → node → phải | `[2, 1, 4, 3, 5]` |
| **post-order** (hậu thứ tự) | trái → phải → node | `[2, 4, 5, 3, 1]` |

Hãy nhớ pre-order: node được ghi **trước** các con, nên khi đọc lại ta gặp root **đầu tiên** — đó sẽ là xương sống của cách chính.

In [6]:
def count_nodes(node):
    if node is None:            # base case: cây rỗng có 0 node
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


def height(node):
    if node is None:            # base case: cây rỗng cao 0
        return 0
    return 1 + max(height(node.left), height(node.right))


def preorder(root):
    out = []
    def go(node):
        if node is None:
            return
        out.append(node.val)    # xử lý node TRƯỚC
        go(node.left)
        go(node.right)
    go(root)
    return out


def inorder(root):
    out = []
    def go(node):
        if node is None:
            return
        go(node.left)
        out.append(node.val)    # xử lý node GIỮA
        go(node.right)
    go(root)
    return out


def postorder(root):
    out = []
    def go(node):
        if node is None:
            return
        go(node.left)
        go(node.right)
        out.append(node.val)    # xử lý node SAU CÙNG
    go(root)
    return out


print("số node :", count_nodes(root1), "| chiều cao:", height(root1))
print("pre-order :", preorder(root1))
print("in-order  :", inorder(root1))
print("post-order:", postorder(root1))

số node : 5 | chiều cao: 3
pre-order : [1, 2, 3, 4, 5]
in-order  : [2, 1, 4, 3, 5]
post-order: [2, 4, 5, 3, 1]


### 3.4 So sánh hai cây: `same_tree` (và bản lặp cho cây sâu)

Để kiểm tra deserialize có đúng không, ta phải so **cây dựng lại** với **cây gốc**. Viết `a == b` với hai `TreeNode` là **không được**: class `TreeNode` không định nghĩa cách so sánh, nên `==` chỉ hỏi "có phải **cùng một object** không". Hai cây giống hệt nhau nhưng gồm các object khác nhau vẫn cho `False`. Ta cần so **cấu trúc** (chính là LeetCode 100 — Same Tree):

- cả hai là `None` → giống nhau;
- chỉ một bên là `None`, hoặc hai `val` khác nhau → khác nhau;
- còn lại: so cây con trái với cây con trái, cây con phải với cây con phải.

Bản đệ quy rất gọn nhưng sẽ vỡ trên cây sâu (mục 3.2). Vì ta sẽ test cây lệch `10^4` tầng, viết thêm bản lặp `same_tree_iter` dùng **stack** (ngăn xếp) tự quản lý: một `list` bình thường, `append` đặt lên **đỉnh**, `pop()` lấy từ **đỉnh** (vào sau ra trước — LIFO). Thay vì để Python nhớ "còn những cặp node nào phải so" trên call stack (bị giới hạn ~1000), ta tự cất các cặp đó vào list của mình — list này nằm ở bộ nhớ thường nên không bị giới hạn đó.

Hàm `make_chain(n, side)` dựng cây lệch (mỗi node chỉ có một con, `side` là `"left"`, `"right"` hoặc `"zigzag"`) bằng vòng lặp. Giá trị lấy `i % 1000` để luôn nằm trong `[-1000, 1000]` như đề.

In [7]:
def same_tree(p, q):
    if p is None and q is None:
        return True
    if p is None or q is None or p.val != q.val:
        return False
    return same_tree(p.left, q.left) and same_tree(p.right, q.right)


def same_tree_iter(p, q):
    stack = [(p, q)]                   # các cặp node còn chờ so sánh
    while stack:
        a, b = stack.pop()
        if a is None and b is None:
            continue
        if a is None or b is None or a.val != b.val:
            return False
        stack.append((a.left, b.left))
        stack.append((a.right, b.right))
    return True


def make_chain(n, side="left"):
    # Cây lệch: mỗi node chỉ có 1 con -> chiều cao = n. Dựng bằng vòng lặp.
    if n == 0:
        return None
    root = TreeNode(0)
    cur = root
    for i in range(1, n):
        child = TreeNode(i % 1000)
        go_left = side == "left" or (side == "zigzag" and i % 2 == 1)
        if go_left:
            cur.left = child
        else:
            cur.right = child
        cur = child
    return root


a, b = build_tree(ex1), build_tree(ex1)
print("a == b         :", a == b)                 # False: hai bộ object khác nhau
print("same_tree(a, b):", same_tree(a, b))         # True : cùng cấu trúc, cùng giá trị
print("khác trái/phải :", same_tree(build_tree([1, 2]), build_tree([1, None, 2])))

print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
c, d = make_chain(3000), make_chain(3000)
try:
    same_tree(c, d)
except RecursionError as e:
    print("same_tree trên cây sâu 3000 ->", type(e).__name__)
print("same_tree_iter trên cây sâu 3000 ->", same_tree_iter(c, d))

a == b         : False
same_tree(a, b): True
khác trái/phải : False
giới hạn đệ quy hiện tại: 1000
same_tree trên cây sâu 3000 -> RecursionError
same_tree_iter trên cây sâu 3000 -> True


### 3.5 Công cụ chuỗi: `join`, `split`, `int`

Mọi cách giải đều biến cây thành **danh sách token** (các mẩu chữ như `"1"`, `"-1000"`, `"#"`) rồi dán lại thành một chuỗi, và làm ngược lại khi đọc:

- `",".join(tokens)`: dán các chuỗi trong list bằng dấu phẩy. Gom token vào list rồi `join` **một lần** vừa nhanh vừa không bị dư dấu phẩy ở cuối.
- `s.split(",")`: cắt chuỗi tại mỗi dấu phẩy, trả về list token.
- `int("-1000")` → `-1000`: `int` hiểu cả dấu âm, nên số âm và số nhiều chữ số không cần xử lý đặc biệt — **miễn là ta tách theo token, không tách theo từng ký tự**.
- **Bẫy:** `"".split(",")` cho `['']` (một list chứa **một chuỗi rỗng**), **không phải** `[]`. Nếu ai đó serialize cây rỗng thành `""`, deserialize sẽ gặp `int('')` và vỡ.
- `s[1:-1]`: bỏ ký tự đầu và ký tự cuối — dùng để gỡ cặp ngoặc `[` `]` của định dạng LeetCode.

### 3.6 `iter` và `next`: "băng chuyền token"

`iter(lst)` tạo một **iterator** — một object **nhớ mình đang đọc tới đâu**. Mỗi lần gọi `next(it)`, nó trả về phần tử kế tiếp và tiến lên một bước; hết phần tử thì ném lỗi `StopIteration`.

Hãy hình dung băng chuyền ở quầy thu ngân: ai lấy thì lấy món **đầu tiên còn lại** trên băng, món đã lấy không bao giờ quay lại.

Điểm quý giá cho bài này: nếu nhiều lần gọi hàm (kể cả các lời gọi đệ quy ở nhiều độ sâu khác nhau) cùng dùng **chung một iterator**, chúng **dùng chung vị trí đọc**. Không cần biến chỉ số `i` truyền qua lại, không cần `nonlocal`, không cần `list.pop(0)` chậm chạp. Trong cell dưới, hàm `read_one` được gọi 3 lần và lấy lần lượt `a`, `b`, `c` — mỗi lần gọi "tiêu thụ" một phần tử khỏi băng chuyền.

In [8]:
tokens = ["1", "2", "#", "-1000"]
s = ",".join(tokens)
print(repr(s))                    # '1,2,#,-1000'
print(s.split(","))               # ['1', '2', '#', '-1000']
print(int("-1000") + 1)           # -999
print(repr(str(-7)), len(str(-1000)))   # '-7' và 5 ký tự: token dài nhất của bài
print("".split(","))              # ['']  <- KHÔNG phải []
print("[1,2,3]"[1:-1])            # 1,2,3

it = iter(["1", "2", "#"])
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("hết token -> StopIteration")


def read_one(stream):
    return next(stream)


stream = iter("abcde")
print(read_one(stream), read_one(stream), read_one(stream))   # a b c: dùng chung vị trí đọc
print("còn lại trên băng chuyền:", list(stream))               # ['d', 'e']

'1,2,#,-1000'
['1', '2', '#', '-1000']
-999
'-7' 5
['']
1,2,3
1 2 #
hết token -> StopIteration
a b c
còn lại trên băng chuyền: ['d', 'e']


## 4. Cách 1 — Cách "lười": để thư viện có sẵn lo phần chuỗi

### 4.1 Ý tưởng: cây → dict lồng nhau → JSON

Người mới thường nghĩ ngay: *"Python có sẵn thư viện biến dữ liệu thành chuỗi rồi, cứ thế mà dùng!"*. **JSON** (JavaScript Object Notation) là định dạng văn bản phổ biến nhất để lưu và gửi dữ liệu. Module `json` có hai hàm:

- `json.dumps(obj)`: dict / list / số / `None` → chuỗi (`None` được ghi thành `null`).
- `json.loads(s)`: chuỗi → dict / list.

JSON không biết `TreeNode` là gì, nên ta đổi mỗi node thành một dict `{"val": ..., "left": ..., "right": ...}`, trong đó `left`/`right` là dict của con (lồng bên trong) hoặc `None`. Việc đổi qua lại là đệ quy rất tự nhiên:

```text
to_dict(None) = None
to_dict(node) = {"val": node.val, "left": to_dict(node.left), "right": to_dict(node.right)}
```

Cách này **đúng**: mỗi dict ghi rõ cả con trái lẫn con phải, **kể cả khi chúng là `None`**, nên không mất thông tin hình dạng. Hãy nhớ chi tiết "ghi cả `None`" này — nó sẽ là chìa khóa ở mục 5.

Độ phức tạp: mỗi node được đổi thành dict đúng một lần, nên `O(n)` thời gian; bộ nhớ `O(n)` cho dict lồng nhau và chuỗi, cộng `O(h)` cho đệ quy.

In [9]:
def tree_to_dict(node):
    if node is None:
        return None
    return {"val": node.val, "left": tree_to_dict(node.left), "right": tree_to_dict(node.right)}


def dict_to_tree(d):
    if d is None:
        return None
    node = TreeNode(d["val"])
    node.left = dict_to_tree(d["left"])
    node.right = dict_to_tree(d["right"])
    return node


def json_serialize(root):
    return json.dumps(tree_to_dict(root))


def json_deserialize(data):
    return dict_to_tree(json.loads(data))


s = json_serialize(root1)
print(s)
print("độ dài:", len(s), "ký tự")
back = json_deserialize(s)
print("dựng lại:", tree_to_list(back), "| same_tree:", same_tree(root1, back), "| cùng object?", back is root1)
print("cây rỗng ->", repr(json_serialize(None)), "->", json_deserialize(json_serialize(None)))

{"val": 1, "left": {"val": 2, "left": null, "right": null}, "right": {"val": 3, "left": {"val": 4, "left": null, "right": null}, "right": {"val": 5, "left": null, "right": null}}}
độ dài: 179 ký tự
dựng lại: [1, 2, 3, None, None, 4, 5] | same_tree: True | cùng object? False
cây rỗng -> 'null' -> None


### 4.2 Biến thể: `repr` của tuple lồng nhau, đọc lại bằng `eval` hay `ast.literal_eval`?

Gọn hơn dict một chút: mỗi node thành tuple `(val, trái, phải)`. Hàm `repr(...)` cho ta một chuỗi đúng cú pháp Python: `(1, (2, None, None), (3, ...))`.

Đọc lại thế nào? Cách "lười nhất" là `eval(s)` — **chạy** chuỗi như một đoạn code Python. Rất **nguy hiểm**: nếu chuỗi đến từ file hay từ mạng do người khác gửi, `eval` sẽ chạy **bất kỳ lệnh nào** nằm trong đó (xóa file, đọc dữ liệu riêng tư...). Cell dưới minh họa bằng một lệnh vô hại là `print`.

An toàn hơn là `ast.literal_eval(s)`: nó chỉ chấp nhận **literal** (số, chuỗi, tuple, list, dict, `None`...) và từ chối mọi lời gọi hàm.

In [10]:
def tree_to_tuple(node):
    if node is None:
        return None
    return (node.val, tree_to_tuple(node.left), tree_to_tuple(node.right))


def tuple_to_tree(t):
    if t is None:
        return None
    node = TreeNode(t[0])
    node.left = tuple_to_tree(t[1])
    node.right = tuple_to_tree(t[2])
    return node


def tuple_serialize(root):
    return repr(tree_to_tuple(root))


def tuple_deserialize(data):
    return tuple_to_tree(ast.literal_eval(data))     # KHÔNG dùng eval


s = tuple_serialize(root1)
print(s, "| độ dài:", len(s))
print("same_tree:", same_tree(root1, tuple_deserialize(s)))

evil = "print('!!! Chuỗi này vừa CHẠY code trên máy bạn !!!') or None"
eval(evil)                                           # eval chạy code nằm trong chuỗi
try:
    ast.literal_eval(evil)
except ValueError as e:
    print("literal_eval từ chối:", type(e).__name__)

(1, (2, None, None), (3, (4, None, None), (5, None, None))) | độ dài: 59
same_tree: True
!!! Chuỗi này vừa CHẠY code trên máy bạn !!!
literal_eval từ chối: ValueError


### 4.3 Cách lười vướng ở đâu?

1. **Dài và thừa.** JSON lặp lại `"val"`, `"left"`, `"right"` ở **mọi** node; Example 1 tốn hơn 170 ký tự trong khi chỉ có 5 con số. (Số liệu so sánh đầy đủ ở mục 13.)
2. **Vỡ trên cây sâu.** `tree_to_dict` / `tree_to_tuple` là đệ quy Python, nên cây lệch sâu hơn khoảng 1000 tầng sẽ gặp `RecursionError`. Ngay cả thư viện cũng có giới hạn độ lồng riêng: `ast.literal_eval` từ chối chuỗi có quá khoảng 200 tầng ngoặc lồng nhau, `json` cũng có giới hạn của nó. Mà đề cho `n` tới `10^4`, tức cây lệch sâu tới `10^4` tầng.
3. **Không học được gì.** Ta chưa hiểu *vì sao* một chuỗi lại đủ để dựng lại cây; phỏng vấn viên muốn thấy bạn **tự thiết kế** định dạng.
4. **`eval` không an toàn** (đã thấy ở trên).

Cell dưới thử round-trip trên cây lệch có độ sâu 150, 300, 1500 và `10^4`. Hàm `try_round_trip` trả `"OK"` nếu dựng lại đúng cây, ngược lại trả **tên lỗi** gặp phải. Tên lỗi cụ thể có thể khác nhau giữa các phiên bản Python; điều quan trọng là **có vỡ hay không**.

In [11]:
def try_round_trip(ser, deser, root):
    # Trả "OK" nếu dựng lại đúng cây, ngược lại trả tên lỗi gặp phải.
    try:
        back = deser(ser(root))
    except Exception as e:              # RecursionError, MemoryError, SyntaxError, ...
        return type(e).__name__
    return "OK" if same_tree_iter(root, back) else "SAI CẤU TRÚC"


print(f"{'độ sâu':>7} | {'JSON':<15} | tuple + literal_eval")
for depth in [150, 300, 1500, 10_000]:
    chain = make_chain(depth, "left")
    r_json = try_round_trip(json_serialize, json_deserialize, chain)
    r_tuple = try_round_trip(tuple_serialize, tuple_deserialize, chain)
    print(f"{depth:>7} | {r_json:<15} | {r_tuple}")

 độ sâu | JSON            | tuple + literal_eval
    150 | OK              | OK
    300 | OK              | MemoryError
   1500 | RecursionError  | RecursionError
  10000 | RecursionError  | RecursionError


## 5. Tự thiết kế định dạng: thử cách đơn giản nhất và xem nó hỏng thế nào

### 5.1 "Chỉ cần ghi các giá trị theo pre-order là đủ?"

Bỏ thư viện, tự nghĩ. Ý tưởng đầu tiên của hầu hết mọi người: duyệt pre-order, ghi các giá trị cách nhau bởi dấu phẩy. Example 1 thành `"1,2,3,4,5"` — chỉ 9 ký tự, gọn hơn JSON gần 20 lần!

Nhưng chỉ với `"1,2,3,4,5"` trong tay, ta có dựng lại được cây không? Thử với hai cây nhỏ nhất có thể:

```text
Cây A:   1          Cây B:   1
        /                     \
       2                       2
```

Pre-order của **cả hai** đều là `"1,2"`. Chuỗi giống hệt nhau, nhưng cây khác nhau. Khi deserialize `"1,2"`, ta không thể biết `2` là con **trái** hay con **phải** — thông tin về hình dạng đã **mất** ngay lúc serialize.

In [12]:
def preorder_no_null(root):
    return ",".join(str(v) for v in preorder(root))


tree_a = build_tree([1, 2])            # 2 là con TRÁI
tree_b = build_tree([1, None, 2])      # 2 là con PHẢI
print("Cây A:"); print_tree(tree_a)
print("Cây B:"); print_tree(tree_b)
print("pre-order không null:", repr(preorder_no_null(tree_a)), "vs", repr(preorder_no_null(tree_b)))
print("hai cây giống nhau? ", same_tree(tree_a, tree_b))

Cây A:
1
├── L: 2
└── R: ·
Cây B:
1
├── L: ·
└── R: 2
pre-order không null: '1,2' vs '1,2'
hai cây giống nhau?  False


### 5.2 Tệ hơn nữa: **mọi** hình dạng cây 3 node đều ra cùng một chuỗi

Liệt kê tất cả hình dạng cây có `n` node: chọn `k` node cho cây con trái (`k = 0, 1, ..., n-1`), `n-1-k` node cho cây con phải, rồi ghép mọi cách dựng bên trái với mọi cách dựng bên phải. Số hình dạng tăng rất nhanh: `1, 1, 2, 5, 14, 42, 132, 429` với `n = 0..7` (đó là dãy **số Catalan**).

Ta biểu diễn một **hình dạng** (chưa có giá trị) bằng tuple `(trái, phải)`, `None` là cây rỗng. Hàm `shape_to_tree` dựng `TreeNode` từ hình dạng và gán giá trị theo pre-order bằng một iterator — dùng lại đúng "băng chuyền" ở mục 3.6.

Nếu gán `1, 2, 3` theo pre-order cho cả 5 hình dạng 3 node, chuỗi pre-order **luôn** là `"1,2,3"`: **5 cây khác nhau, 1 chuỗi**. Không hàm deserialize nào cứu được: nhận `"1,2,3"` nó chỉ trả về được **một** cây, trong khi có 5 đáp án đúng khác nhau. Lý luận y hệt cho thấy chỉ in-order, chỉ post-order, hay level-order mà bỏ `null` cũng đều không đủ: chuỗi chỉ chứa **giá trị**, không chứa **hình dạng**.

In [13]:
def all_shapes(n):
    # Mọi hình dạng cây n node; mỗi hình dạng là tuple lồng (trái, phải), None = cây rỗng.
    if n == 0:
        return [None]
    shapes = []
    for k in range(n):                       # k node bên trái, n-1-k node bên phải
        for left in all_shapes(k):
            for right in all_shapes(n - 1 - k):
                shapes.append((left, right))
    return shapes


def shape_to_tree(shape, values):
    # Dựng TreeNode mới theo hình dạng; giá trị lấy lần lượt từ iterator `values` theo pre-order.
    if shape is None:
        return None
    node = TreeNode(next(values))
    node.left = shape_to_tree(shape[0], values)
    node.right = shape_to_tree(shape[1], values)
    return node


print("số hình dạng n = 0..7:", [len(all_shapes(n)) for n in range(8)])
shapes3 = [shape_to_tree(sh, iter([1, 2, 3])) for sh in all_shapes(3)]
for t in shapes3:
    print(f"{str(tree_to_list(t)):<26} pre-order không null = {preorder_no_null(t)!r}")

số hình dạng n = 0..7: [1, 1, 2, 5, 14, 42, 132, 429]
[1, None, 2, None, 3]      pre-order không null = '1,2,3'
[1, None, 2, 3]            pre-order không null = '1,2,3'
[1, 2, 3]                  pre-order không null = '1,2,3'
[1, 2, None, None, 3]      pre-order không null = '1,2,3'
[1, 2, None, 3]            pre-order không null = '1,2,3'


### 5.3 Mấu chốt: ghi cả **chỗ trống**

JSON ở mục 4 đúng là vì nó ghi cả `None`. Vậy giữ đúng ý đó nhưng bỏ phần rườm rà: vẫn duyệt pre-order, nhưng **mỗi khi gặp `None` thì ghi một ký hiệu ngắn** `#` thay vì bỏ qua. Hai cây ở mục 5.1 giờ khác nhau rõ ràng:

```text
Cây A (2 là con trái):  1,2,#,#,#     <- sau 1 là 2 (trái), rồi hai # của 2, rồi # là con phải của 1
Cây B (2 là con phải):  1,#,2,#,#     <- sau 1 là # (trái trống), rồi 2 (phải), rồi hai # của 2
```

**Ghi chỗ trống tốn bao nhiêu?** Một cây `n` node có **đúng `n + 1` chỗ trống** (ô con bằng `None`). Lý do: `n` node có tổng cộng `2n` ô con; mỗi node trừ root nằm ở đúng một ô con của cha nó, nên có `n - 1` ô đã có node; còn lại `2n - (n - 1) = n + 1` ô trống.

Kiểm tra với Example 1 (`n = 5`): có `10` ô con; `4` ô có node (`2`, `3`, `4`, `5`); còn `6` ô trống — 2 ô của node `2`, 2 ô của `4`, 2 ô của `5`. Đúng `5 + 1 = 6`.

Vậy chuỗi có `n` giá trị + `(n + 1)` dấu `#` = **`2n + 1` token** (Example 1: 11 token). Chỉ dài cỡ gấp đôi, vẫn là `O(n)`.

Vì sao chọn `#`? Vì giá trị trong cây là số nguyên, không bao giờ chứa `#` hay `,` — nên ký hiệu chỗ trống và dấu phân cách **không thể bị nhầm** với một phần của giá trị.

In [14]:
def count_empty_slots(node):
    if node is None:
        return 1                         # bản thân ô này là một chỗ trống
    return count_empty_slots(node.left) + count_empty_slots(node.right)


print("Example 1: n =", count_nodes(root1), "| chỗ trống =", count_empty_slots(root1))
for n in range(0, 8):
    for sh in all_shapes(n):
        t = shape_to_tree(sh, iter(range(n)))
        assert count_empty_slots(t) == n + 1
print("Mọi hình dạng n = 0..7 đều có đúng n + 1 chỗ trống.")

Example 1: n = 5 | chỗ trống = 6
Mọi hình dạng n = 0..7 đều có đúng n + 1 chỗ trống.


## 6. Cách 2 (cách chính) — DFS pre-order có ký hiệu `#`

### 6.1 Serialize

Quy tắc viết bằng đệ quy:

```text
ser(None) = "#"
ser(node) = str(node.val) , ser(node.left) , ser(node.right)     (nối bằng dấu phẩy)
```

Trong code, thay vì cộng chuỗi (chậm, dễ dư dấu phẩy), ta gom token vào một list `tokens` rồi `",".join` một lần ở cuối. Hàm con `walk` nằm **bên trong** `dfs_serialize` nên dùng thẳng được biến `tokens` của hàm ngoài (gọi `tokens.append` là **sửa** list, không phải gán lại biến, nên không cần `nonlocal`).

Tính tay Example 1:

```text
ser(1) = 1 , ser(2)  , ser(3)
       = 1 , 2,#,#   , 3 , ser(4) , ser(5)
       = 1 , 2,#,#   , 3 , 4,#,#  , 5,#,#
       = "1,2,#,#,3,4,#,#,5,#,#"
```

Đếm: 5 giá trị + 6 dấu `#` = 11 token = `2·5 + 1`. Cây rỗng cho `"#"`; cây một node `[7]` cho `"7,#,#"`. Cell dưới cũng chạy lại 5 hình dạng 3 node của mục 5.2: giờ 5 cây cho **5 chuỗi khác nhau**.

In [15]:
def dfs_serialize(root):
    tokens = []

    def walk(node):
        if node is None:
            tokens.append("#")          # chỗ trống
            return
        tokens.append(str(node.val))    # node trước...
        walk(node.left)                 # ...rồi toàn bộ cây con trái...
        walk(node.right)                # ...rồi toàn bộ cây con phải

    walk(root)
    return ",".join(tokens)


print(dfs_serialize(root1))
print(repr(dfs_serialize(None)), repr(dfs_serialize(build_tree([7]))))
print("Cây A:", dfs_serialize(tree_a), "| Cây B:", dfs_serialize(tree_b))
print()
for t in shapes3:
    print(f"{str(tree_to_list(t)):<26} có '#' = {dfs_serialize(t)}")
print("số chuỗi khác nhau:", len({dfs_serialize(t) for t in shapes3}), "/ 5")

1,2,#,#,3,4,#,#,5,#,#
'#' '7,#,#'
Cây A: 1,2,#,#,# | Cây B: 1,#,2,#,#

[1, None, 2, None, 3]      có '#' = 1,#,2,#,3,#,#
[1, None, 2, 3]            có '#' = 1,#,2,3,#,#,#
[1, 2, 3]                  có '#' = 1,2,#,#,3,#,#
[1, 2, None, None, 3]      có '#' = 1,2,#,3,#,#,#
[1, 2, None, 3]            có '#' = 1,2,3,#,#,#,#
số chuỗi khác nhau: 5 / 5


### 6.2 Deserialize: đọc token theo đúng nhịp đã ghi

Chuỗi được ghi theo nhịp *node → toàn bộ cây con trái → toàn bộ cây con phải*, nên đọc lại cũng đúng nhịp đó:

```text
build():
    tok = token kế tiếp
    nếu tok == "#": trả về None             # chỗ trống
    node = TreeNode(int(tok))
    node.left  = build()                    # các token kế tiếp mô tả TOÀN BỘ cây con trái
    node.right = build()                    # tiếp theo là toàn bộ cây con phải
    trả về node
```

**Câu hỏi then chốt:** lời gọi `build()` cho cây con trái làm sao biết **dừng ở đâu**, để phần còn lại thuộc về cây con phải? Nhờ `#`: mỗi nhánh của cây con trái đều kết thúc bằng `#`, nên lời gọi đó tự dừng **đúng sau token cuối cùng** của cây con trái — không cần biết trước cây con có bao nhiêu node. Không có `#` (mục 5) thì không có "điểm dừng", và mọi thứ trở nên mơ hồ.

**Vì sao dùng `iter` + `next`?** Mọi lời gọi `build()` ở mọi độ sâu phải đọc trên **cùng một** dòng token và cùng tiến lên. Một iterator dùng chung làm đúng việc đó (mục 3.6). Các lựa chọn khác cũng đúng: một chỉ số `i` dùng chung (`nonlocal i` hoặc `self.i`), `deque.popleft()`, hay đảo ngược list rồi `pop()` từ cuối. Chỉ cần tránh `list.pop(0)` vì mỗi lần tốn `O(n)` (mục 15).

**Bất biến (invariant) cần nhớ:** *mỗi lời gọi `build()` đọc đúng các token của một cây con — không thừa, không thiếu — và trả về đúng cây con đó.* Chứng minh đầy đủ ở mục 14; trace ở mục 7 cho thấy nó hoạt động.

**Độ phức tạp cách 2:** serialize thăm mỗi node một lần và ghi `2n + 1` token; deserialize đọc mỗi token đúng một lần → **`O(n)` thời gian** cho cả hai chiều. Bộ nhớ: chuỗi và list token `O(n)` (không tránh được, đầu ra đã dài cỡ đó), cộng call stack **`O(h)`** — `O(log n)` với cây cân bằng, `O(n)` với cây lệch.

In [16]:
def dfs_deserialize(data):
    tokens = iter(data.split(","))       # băng chuyền token dùng chung

    def build():
        tok = next(tokens)
        if tok == "#":
            return None
        node = TreeNode(int(tok))
        node.left = build()              # đọc trọn cây con trái
        node.right = build()             # rồi trọn cây con phải
        return node

    return build()


back = dfs_deserialize("1,2,#,#,3,4,#,#,5,#,#")
print_tree(back)
print("list:", tree_to_list(back), "| same_tree với root1:", same_tree(back, root1))
print("deserialize('#') ->", dfs_deserialize("#"))
for vals in [[], [7], [0, 0, 0], [-1000, 1000], [1, None, 2, 3], [5, 4, 7, 3, None, 2, None, -1, None, 9]]:
    t = build_tree(vals)
    assert same_tree(dfs_deserialize(dfs_serialize(t)), t)
print("round-trip DFS OK trên các cây mẫu")

1
├── L: 2
└── R: 3
    ├── L: 4
    └── R: 5
list: [1, 2, 3, None, None, 4, 5] | same_tree với root1: True
deserialize('#') -> None
round-trip DFS OK trên các cây mẫu


## 7. Trace từng bước cách chính

### 7.1 Serialize Example 1

Mỗi dòng là một lần gọi `walk`, thụt lề theo độ sâu; cột cuối là chuỗi sau khi ghi.

| Bước | `walk` tại | Ghi | Chuỗi sau bước |
|---:|---|:---:|---|
| 1 | root: node `1` | `1` | `1` |
| 2 | trái của 1: node `2` | `2` | `1,2` |
| 3 | trái của 2: `None` | `#` | `1,2,#` |
| 4 | phải của 2: `None` | `#` | `1,2,#,#` |
| 5 | phải của 1: node `3` | `3` | `1,2,#,#,3` |
| 6 | trái của 3: node `4` | `4` | `1,2,#,#,3,4` |
| 7 | trái của 4: `None` | `#` | `1,2,#,#,3,4,#` |
| 8 | phải của 4: `None` | `#` | `1,2,#,#,3,4,#,#` |
| 9 | phải của 3: node `5` | `5` | `1,2,#,#,3,4,#,#,5` |
| 10 | trái của 5: `None` | `#` | `1,2,#,#,3,4,#,#,5,#` |
| 11 | phải của 5: `None` | `#` | `1,2,#,#,3,4,#,#,5,#,#` |

Để ý: mỗi node được ghi **trước** các con của nó; mỗi **lá** sinh ra đúng hai dấu `#` liền sau nó; toàn bộ cây con gốc `2` là đoạn liền `2,#,#`, cây con gốc `3` là đoạn liền `3,4,#,#,5,#,#`.

In [17]:
def trace_dfs_serialize(root):
    tokens = []

    def walk(node, depth, where):
        pad = "    " * depth
        if node is None:
            tokens.append("#")
            print(f"{pad}{where}: None -> ghi '#'".ljust(42) + "| " + ",".join(tokens))
            return
        tokens.append(str(node.val))
        print(f"{pad}{where}: node {node.val} -> ghi '{node.val}'".ljust(42) + "| " + ",".join(tokens))
        walk(node.left, depth + 1, f"trái của {node.val}")
        walk(node.right, depth + 1, f"phải của {node.val}")

    walk(root, 0, "root")
    return ",".join(tokens)


result = trace_dfs_serialize(root1)
print("\nKết quả:", result)

root: node 1 -> ghi '1'                   | 1
    trái của 1: node 2 -> ghi '2'         | 1,2
        trái của 2: None -> ghi '#'       | 1,2,#
        phải của 2: None -> ghi '#'       | 1,2,#,#
    phải của 1: node 3 -> ghi '3'         | 1,2,#,#,3
        trái của 3: node 4 -> ghi '4'     | 1,2,#,#,3,4
            trái của 4: None -> ghi '#'   | 1,2,#,#,3,4,#
            phải của 4: None -> ghi '#'   | 1,2,#,#,3,4,#,#
        phải của 3: node 5 -> ghi '5'     | 1,2,#,#,3,4,#,#,5
            trái của 5: None -> ghi '#'   | 1,2,#,#,3,4,#,#,5,#
            phải của 5: None -> ghi '#'   | 1,2,#,#,3,4,#,#,5,#,#

Kết quả: 1,2,#,#,3,4,#,#,5,#,#


### 7.2 Deserialize: Example 1 và một cây "khó" hơn

**Trace 1** — `"1,2,#,#,3,4,#,#,5,#,#"`. Cột "Đang chờ" là call stack: các node đã tạo nhưng **chưa xong** (còn chờ dựng con).

| Bước | Token đọc | Vị trí đang dựng | Hành động | Đang chờ (call stack) |
|---:|:---:|---|---|---|
| 1 | `1` | root | tạo node 1, đi dựng con trái của 1 | `[1]` |
| 2 | `2` | trái của 1 | tạo node 2, đi dựng con trái của 2 | `[1, 2]` |
| 3 | `#` | trái của 2 | `None` | `[1, 2]` |
| 4 | `#` | phải của 2 | `None` → **node 2 xong**, trả lên cho 1 | `[1]` |
| 5 | `3` | phải của 1 | tạo node 3 | `[1, 3]` |
| 6 | `4` | trái của 3 | tạo node 4 | `[1, 3, 4]` |
| 7 | `#` | trái của 4 | `None` | `[1, 3, 4]` |
| 8 | `#` | phải của 4 | `None` → **node 4 xong** | `[1, 3]` |
| 9 | `5` | phải của 3 | tạo node 5 | `[1, 3, 5]` |
| 10 | `#` | trái của 5 | `None` | `[1, 3, 5]` |
| 11 | `#` | phải của 5 | `None` → **5 xong** → **3 xong** → **1 xong** | `[]` |

Sau bước 4, lời gọi dựng con trái của 1 đã đọc **đúng 3 token** `2,#,#` và dừng — token kế tiếp `3` tự nhiên thuộc về con phải của 1. Đó chính là bất biến ở mục 6.2. Hết 11 token thì call stack cũng rỗng: không thừa, không thiếu token nào.

**Trace 2** — `"1,#,2,3,#,#,#"` (một trong 5 hình dạng ở mục 5.2: 1 có con **phải** 2, 2 có con **trái** 3). Điểm đáng xem: token thứ 2 là `#` nên con trái của 1 rỗng **ngay lập tức**, và dấu `#` cuối cùng là con phải của 2 — nó chỉ được đọc **sau khi** cả cây con gốc 3 đã xong.

| Bước | Token | Vị trí | Hành động |
|---:|:---:|---|---|
| 1 | `1` | root | tạo node 1 |
| 2 | `#` | trái của 1 | `None` (trái rỗng ngay) |
| 3 | `2` | phải của 1 | tạo node 2 |
| 4 | `3` | trái của 2 | tạo node 3 |
| 5 | `#` | trái của 3 | `None` |
| 6 | `#` | phải của 3 | `None` → node 3 xong |
| 7 | `#` | phải của 2 | `None` → node 2 xong → node 1 xong |

Hàm `trace_dfs_deserialize` dưới in đúng các bước này. Mẹo nhỏ: `iter(enumerate(...))` cho băng chuyền các cặp `(chỉ số, token)`, nên có số thứ tự bước mà không cần biến đếm riêng.

In [18]:
def trace_dfs_deserialize(data):
    tokens = iter(enumerate(data.split(",")))      # băng chuyền các cặp (chỉ số, token)

    def build(depth, where):
        i, tok = next(tokens)
        pad = "    " * depth
        if tok == "#":
            print(f"{i + 1:>2}. {pad}đọc '#' -> {where} = None")
            return None
        node = TreeNode(int(tok))
        print(f"{i + 1:>2}. {pad}đọc '{tok}' -> tạo node {tok} ({where})")
        node.left = build(depth + 1, f"trái của {tok}")
        node.right = build(depth + 1, f"phải của {tok}")
        print(f"    {pad}xong node {tok}" + (" -> trả về làm root" if depth == 0 else " -> trả lên cho cha"))
        return node

    root = build(0, "root")
    print("token còn thừa:", [tok for _, tok in tokens])
    return root


print("=== Trace 1 ===")
t1 = trace_dfs_deserialize("1,2,#,#,3,4,#,#,5,#,#")
print("\n=== Trace 2 ===")
t2 = trace_dfs_deserialize("1,#,2,3,#,#,#")
print_tree(t2)

=== Trace 1 ===
 1. đọc '1' -> tạo node 1 (root)
 2.     đọc '2' -> tạo node 2 (trái của 1)
 3.         đọc '#' -> trái của 2 = None
 4.         đọc '#' -> phải của 2 = None
        xong node 2 -> trả lên cho cha
 5.     đọc '3' -> tạo node 3 (phải của 1)
 6.         đọc '4' -> tạo node 4 (trái của 3)
 7.             đọc '#' -> trái của 4 = None
 8.             đọc '#' -> phải của 4 = None
            xong node 4 -> trả lên cho cha
 9.         đọc '5' -> tạo node 5 (phải của 3)
10.             đọc '#' -> trái của 5 = None
11.             đọc '#' -> phải của 5 = None
            xong node 5 -> trả lên cho cha
        xong node 3 -> trả lên cho cha
    xong node 1 -> trả về làm root
token còn thừa: []

=== Trace 2 ===
 1. đọc '1' -> tạo node 1 (root)
 2.     đọc '#' -> trái của 1 = None
 3.     đọc '2' -> tạo node 2 (phải của 1)
 4.         đọc '3' -> tạo node 3 (trái của 2)
 5.             đọc '#' -> trái của 3 = None
 6.             đọc '#' -> phải của 3 = None
            xong node 3 

## 8. Code nộp LeetCode (cách chính)

Chuyển hai hàm `dfs_serialize` / `dfs_deserialize` vào đúng khung `Codec`:

- Giữ nguyên tên class `Codec`, tên method `serialize` / `deserialize` và docstring của template.
- Các hàm con `walk` / `build` đặt **bên trong** method, dùng được biến `tokens` của method (closure — "hàm con nhớ biến của hàm bao quanh").
- `TreeNode(int(tok))` — đúng **một** tham số như định nghĩa của bài 297.
- Cây rỗng tự động đúng: `serialize(None)` trả `"#"`, `deserialize("#")` trả `None` — không cần `if` riêng.
- `Codec` **không lưu gì trong `self`**, nên việc LeetCode dùng hai object `ser` và `deser` khác nhau không ảnh hưởng.
- Về độ sâu: đây là bản đệ quy, độ sâu đệ quy bằng chiều cao cây. Với cây lệch thật sâu (tới `10^4` tầng theo constraints) cần tăng giới hạn đệ quy hoặc dùng bản lặp — xem mục 11 (bản BFS ở mục 9 thì không có vấn đề này).

In [19]:
class Codec:

    def serialize(self, root):
        """Encodes a tree to a single string.

        :type root: TreeNode
        :rtype: str
        """
        tokens = []

        def walk(node):
            if node is None:
                tokens.append("#")
                return
            tokens.append(str(node.val))
            walk(node.left)
            walk(node.right)

        walk(root)
        return ",".join(tokens)

    def deserialize(self, data):
        """Decodes your encoded data to tree.

        :type data: str
        :rtype: TreeNode
        """
        tokens = iter(data.split(","))

        def build():
            tok = next(tokens)
            if tok == "#":
                return None
            node = TreeNode(int(tok))
            node.left = build()
            node.right = build()
            return node

        return build()


# Your Codec object will be instantiated and called as such:
ser = Codec()
deser = Codec()
ans = deser.deserialize(ser.serialize(build_tree(ex1)))
print(tree_to_list(ans))

[1, 2, 3, None, None, 4, 5]


### 8.2 Mô phỏng đúng cách LeetCode chấm

Nhiều bài thiết kế class (như MedianFinder) được test bằng chuỗi lệnh kiểu `["MedianFinder","addNum",...]`. Bài 297 thì không: "chuỗi lệnh" chính là 3 dòng comment cuối template. Hàm `run_codec_leetcode(codec_cls, values)` làm y hệt:

1. `root = build_tree(values)` — LeetCode dựng cây từ Input.
2. `ser = codec_cls()` và `deser = codec_cls()` — **hai object khác nhau**.
3. `ans = deser.deserialize(ser.serialize(root))`.
4. Trả về `tree_to_list(ans)` — LeetCode in `Output` dạng list, phải bằng Input.

Ta kiểm tra thêm `ans is not root` (cây phải được **dựng mới**, không phải trả lại chính object cũ) và `same_tree(ans, root)`. Bộ test gồm hai ví dụ của đề và các edge case: một node, giá trị `0`, biên giá trị `-1000`/`1000`, toàn số âm, giá trị trùng, cây lệch trái, lệch phải, và một cây có `null` nằm giữa.

In [20]:
def run_codec_leetcode(codec_cls, values):
    root = build_tree(values)
    ser = codec_cls()
    deser = codec_cls()
    ans = deser.deserialize(ser.serialize(root))
    assert root is None or ans is not root, "phải dựng cây MỚI"
    assert same_tree(ans, root)
    return tree_to_list(ans)


tests = [
    [1, 2, 3, None, None, 4, 5],                 # Example 1
    [],                                          # Example 2
    [7],                                         # một node
    [0, 0, 0],                                   # giá trị 0
    [-1000, 1000],                               # biên giá trị
    [-1, -2, -3, -4, None, -5],                  # toàn số âm
    [1, 1, 1, None, 1],                          # giá trị trùng
    [1, 2, None, 3, None, 4],                    # lệch trái
    [1, None, 2, None, 3, None, 4],              # lệch phải
    [5, 4, 7, 3, None, 2, None, -1, None, 9],    # null nằm giữa
]
for vals in tests:
    out = run_codec_leetcode(Codec, vals)
    print(f"Input: {str(vals):<42} Output: {out}")
    assert out == vals
print("Codec (DFS) PASS toàn bộ test kiểu LeetCode!")

Input: [1, 2, 3, None, None, 4, 5]                Output: [1, 2, 3, None, None, 4, 5]
Input: []                                         Output: []
Input: [7]                                        Output: [7]
Input: [0, 0, 0]                                  Output: [0, 0, 0]
Input: [-1000, 1000]                              Output: [-1000, 1000]
Input: [-1, -2, -3, -4, None, -5]                 Output: [-1, -2, -3, -4, None, -5]
Input: [1, 1, 1, None, 1]                         Output: [1, 1, 1, None, 1]
Input: [1, 2, None, 3, None, 4]                   Output: [1, 2, None, 3, None, 4]
Input: [1, None, 2, None, 3, None, 4]             Output: [1, None, 2, None, 3, None, 4]
Input: [5, 4, 7, 3, None, 2, None, -1, None, 9]   Output: [5, 4, 7, 3, None, 2, None, -1, None, 9]
Codec (DFS) PASS toàn bộ test kiểu LeetCode!


## 9. Cách 3 — BFS level-order: đúng định dạng của LeetCode

### 9.1 Ý tưởng

`Output: [1,2,3,null,null,4,5]` của Example 1 chính là kết quả của việc serialize **theo tầng**. Duyệt theo tầng gọi là **BFS** (Breadth-First Search — duyệt theo chiều rộng): xong tầng 0 mới tới tầng 1, xong tầng 1 mới tới tầng 2... Công cụ là **queue** (mục 2.3).

Serialize:

```text
queue = [root]
lặp khi queue còn phần tử:
    node = lấy ra ở ĐẦU queue
    nếu node là None: ghi "null"          (None không có con, không xếp thêm gì)
    ngược lại: ghi node.val, rồi xếp node.left và node.right vào CUỐI queue — kể cả khi chúng là None!
cuối cùng: cắt các "null" thừa ở cuối, bọc trong [ ]
```

Vì sao **được phép** cắt `null` ở cuối? Khi đọc lại, nếu hết token thì mọi ô con còn lại hiển nhiên là `None` (mục 9.3). Với Example 1, bản chưa cắt là `[1,2,3,null,null,4,5,null,null,null,null]` — 4 `null` cuối là con của hai lá `4` và `5`, bỏ đi không mất thông tin. Cây rỗng ghi là `[]`.

Cũng như cách 2, bản chưa cắt luôn có đúng `2n + 1` token (`n` giá trị + `n + 1` chỗ trống). Bản đã cắt thì ngắn hơn — rất ngắn với cây "đầy đủ" (mục 13).

Ưu điểm lớn: BFS chỉ dùng **vòng lặp + queue**, không đệ quy, nên cây sâu `10^4` tầng không phải là vấn đề.

**Độ phức tạp cách 3:** mỗi node vào và ra khỏi queue một lần, mỗi token được ghi/đọc một lần → `O(n)` thời gian. Bộ nhớ phụ là queue, chứa nhiều nhất cỡ **một tầng** của cây (gọi là `w`, độ rộng lớn nhất) — `O(w)`, tệ nhất `O(n)`.

In [21]:
def bfs_serialize(root, trim=True):
    if root is None:
        return "[]"
    out = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append("null")
            continue
        out.append(str(node.val))
        queue.append(node.left)          # xếp cả con None
        queue.append(node.right)
    if trim:
        while out and out[-1] == "null": # cắt null thừa ở cuối
            out.pop()
    return "[" + ",".join(out) + "]"


print(bfs_serialize(root1, trim=False))
print(bfs_serialize(root1))
print(bfs_serialize(None), bfs_serialize(build_tree([1, None, 2, None, 3])))

[1,2,3,null,null,4,5,null,null,null,null]
[1,2,3,null,null,4,5]
[] [1,null,2,null,3]


### 9.2 Trace serialize BFS trên Example 1

| Bước | Lấy ra | Ghi | Xếp thêm | Queue sau bước |
|---:|:---:|:---:|---|---|
| 1 | `1` | `1` | 2, 3 | `[2, 3]` |
| 2 | `2` | `2` | None, None | `[3, None, None]` |
| 3 | `3` | `3` | 4, 5 | `[None, None, 4, 5]` |
| 4 | `None` | `null` | — | `[None, 4, 5]` |
| 5 | `None` | `null` | — | `[4, 5]` |
| 6 | `4` | `4` | None, None | `[5, None, None]` |
| 7 | `5` | `5` | None, None | `[None, None, None, None]` |
| 8–11 | `None` ×4 | `null` ×4 | — | ... → `[]` |

Chưa cắt: `[1,2,3,null,null,4,5,null,null,null,null]` (11 token). Đã cắt: `[1,2,3,null,null,4,5]` (7 token) — trùng khít Output của đề.

In [22]:
def trace_bfs_serialize(root):
    out = []
    queue = deque([root])

    def show(q):
        return "[" + ", ".join("None" if x is None else str(x.val) for x in q) + "]"

    step = 0
    while queue:
        node = queue.popleft()
        step += 1
        if node is None:
            out.append("null")
            print(f"{step:>2}. lấy None -> ghi null".ljust(34) + f"queue = {show(queue)}")
            continue
        out.append(str(node.val))
        queue.append(node.left)
        queue.append(node.right)
        print(f"{step:>2}. lấy {node.val} -> ghi {node.val}, xếp 2 con".ljust(34) + f"queue = {show(queue)}")
    print("chưa cắt:", "[" + ",".join(out) + "]")
    while out and out[-1] == "null":
        out.pop()
    print("đã cắt  :", "[" + ",".join(out) + "]")


trace_bfs_serialize(root1)

 1. lấy 1 -> ghi 1, xếp 2 con     queue = [2, 3]
 2. lấy 2 -> ghi 2, xếp 2 con     queue = [3, None, None]
 3. lấy 3 -> ghi 3, xếp 2 con     queue = [None, None, 4, 5]
 4. lấy None -> ghi null          queue = [None, 4, 5]
 5. lấy None -> ghi null          queue = [4, 5]
 6. lấy 4 -> ghi 4, xếp 2 con     queue = [5, None, None]
 7. lấy 5 -> ghi 5, xếp 2 con     queue = [None, None, None, None]
 8. lấy None -> ghi null          queue = [None, None, None]
 9. lấy None -> ghi null          queue = [None, None]
10. lấy None -> ghi null          queue = [None]
11. lấy None -> ghi null          queue = []
chưa cắt: [1,2,3,null,null,4,5,null,null,null,null]
đã cắt  : [1,2,3,null,null,4,5]


### 9.3 Deserialize BFS bằng queue

Mấu chốt: lúc serialize, các node thật được lấy khỏi queue theo một thứ tự nhất định, và **node thật thứ `k`** (đếm từ 1) ghi hai con của nó vào **hai token liên tiếp**, ở vị trí `2k - 1` và `2k` (root ở vị trí 0). Deserialize chỉ cần **phát lại đúng thứ tự đó**:

- `queue` chứa các node thật **đã tạo nhưng chưa được gắn con**, theo đúng thứ tự gặp.
- `i` là vị trí token **tiếp theo chưa dùng** (bắt đầu từ 1).
- Mỗi lượt: lấy node đầu queue; `tokens[i]` là con trái, `tokens[i + 1]` là con phải; con nào thật thì tạo node và xếp vào cuối queue.
- **Hết token thì dừng**: các node còn trong queue giữ nguyên `left = right = None` (giá trị mặc định của `TreeNode`). Đó chính là lý do việc cắt `null` cuối là an toàn.

Đây thực ra chính là `build_tree` ở mục 2.4, chỉ khác là đọc từ chuỗi thay vì từ list Python.

**Trace 1** — `[1,2,3,null,null,4,5]`, `tokens` có 7 phần tử:

| Lượt | Lấy node | Trái = `tokens[i]` | Phải = `tokens[i+1]` | Queue sau | `i` sau |
|---|:---:|---|---|---|---:|
| khởi tạo | — | tạo root `1` | | `[1]` | 1 |
| 1 | `1` | `tokens[1]=2` → tạo 2 | `tokens[2]=3` → tạo 3 | `[2, 3]` | 3 |
| 2 | `2` | `tokens[3]=null` | `tokens[4]=null` | `[3]` | 5 |
| 3 | `3` | `tokens[5]=4` → tạo 4 | `tokens[6]=5` → tạo 5 | `[4, 5]` | 7 |
| dừng | | `i = 7 = len(tokens)` | | `4`, `5` giữ con `None` | |

**Trace 2** — `[5,4,7,3,null,2,null,-1,null,9]` (có `null` ở giữa và đã bị cắt ở cuối). Ở lượt cuối, node `2` lấy được con trái `9` nhưng **hết token** trước khi tới con phải, nên con phải giữ `None`; hai node `-1` và `9` còn trong queue cũng giữ `None`.

In [23]:
def bfs_deserialize(data):
    body = data.strip()[1:-1]                    # bỏ "[" và "]"
    if body == "":
        return None                              # "[]" -> cây rỗng
    tokens = body.split(",")
    root = TreeNode(int(tokens[0]))
    queue = deque([root])
    i = 1
    while queue and i < len(tokens):
        node = queue.popleft()
        if tokens[i] != "null":                  # con trái
            node.left = TreeNode(int(tokens[i]))
            queue.append(node.left)
        i += 1
        if i < len(tokens) and tokens[i] != "null":   # con phải (có thể đã hết token)
            node.right = TreeNode(int(tokens[i]))
            queue.append(node.right)
        i += 1
    return root


def trace_bfs_deserialize(data):
    tokens = data.strip()[1:-1].split(",")
    print("tokens =", tokens, "| len =", len(tokens))
    root = TreeNode(int(tokens[0]))
    queue = deque([root])
    i = 1
    print(f"khởi tạo: root = {root.val}, queue = [{root.val}], i = 1")
    while queue and i < len(tokens):
        node = queue.popleft()
        parts = []
        for side in ("trái", "phải"):
            if i >= len(tokens):
                parts.append(f"{side}: hết token")
            elif tokens[i] == "null":
                parts.append(f"{side}: tokens[{i}]=null")
            else:
                child = TreeNode(int(tokens[i]))
                if side == "trái":
                    node.left = child
                else:
                    node.right = child
                queue.append(child)
                parts.append(f"{side}: tokens[{i}]={tokens[i]}")
            i += 1
        print(f"lấy {node.val:>2} | " + " | ".join(parts) + f" | queue = {[x.val for x in queue]} | i = {i}")
    print(f"dừng vì i = {i} >= len(tokens) = {len(tokens)}; node còn trong queue giữ con None")
    return root


t = trace_bfs_deserialize("[1,2,3,null,null,4,5]")
print("same_tree với root1:", same_tree(t, root1))
print()
t = trace_bfs_deserialize("[5,4,7,3,null,2,null,-1,null,9]")
print_tree(t)

tokens = ['1', '2', '3', 'null', 'null', '4', '5'] | len = 7
khởi tạo: root = 1, queue = [1], i = 1
lấy  1 | trái: tokens[1]=2 | phải: tokens[2]=3 | queue = [2, 3] | i = 3
lấy  2 | trái: tokens[3]=null | phải: tokens[4]=null | queue = [3] | i = 5
lấy  3 | trái: tokens[5]=4 | phải: tokens[6]=5 | queue = [4, 5] | i = 7
dừng vì i = 7 >= len(tokens) = 7; node còn trong queue giữ con None
same_tree với root1: True

tokens = ['5', '4', '7', '3', 'null', '2', 'null', '-1', 'null', '9'] | len = 10
khởi tạo: root = 5, queue = [5], i = 1
lấy  5 | trái: tokens[1]=4 | phải: tokens[2]=7 | queue = [4, 7] | i = 3
lấy  4 | trái: tokens[3]=3 | phải: tokens[4]=null | queue = [7, 3] | i = 5
lấy  7 | trái: tokens[5]=2 | phải: tokens[6]=null | queue = [3, 2] | i = 7
lấy  3 | trái: tokens[7]=-1 | phải: tokens[8]=null | queue = [2, -1] | i = 9
lấy  2 | trái: tokens[9]=9 | phải: hết token | queue = [-1, 9] | i = 11
dừng vì i = 11 >= len(tokens) = 10; node còn trong queue giữ con None
5
├── L: 4
│   ├── L: 3
│

### 9.4 Bản nộp LeetCode của cách BFS

Trong notebook đặt tên `CodecBFS` để không đè lên `Codec` của cách chính; **khi nộp, đổi tên thành `Codec`**. Bản này hoàn toàn không đệ quy. Ta chạy lại đúng bộ test kiểu LeetCode ở mục 8.2, và kiểm tra thêm: chuỗi do `CodecBFS` sinh ra **trùng khít** với cách LeetCode in Input (định dạng `[...]`, chữ `null`, đã cắt `null` cuối).

In [24]:
class CodecBFS:

    def serialize(self, root):
        if root is None:
            return "[]"
        out, queue = [], deque([root])
        while queue:
            node = queue.popleft()
            if node is None:
                out.append("null")
                continue
            out.append(str(node.val))
            queue.append(node.left)
            queue.append(node.right)
        while out[-1] == "null":                 # root thật nên out không bao giờ rỗng ở đây
            out.pop()
        return "[" + ",".join(out) + "]"

    def deserialize(self, data):
        body = data[1:-1]
        if not body:
            return None
        tokens = body.split(",")
        root = TreeNode(int(tokens[0]))
        queue, i = deque([root]), 1
        while queue and i < len(tokens):
            node = queue.popleft()
            if tokens[i] != "null":
                node.left = TreeNode(int(tokens[i]))
                queue.append(node.left)
            i += 1
            if i < len(tokens) and tokens[i] != "null":
                node.right = TreeNode(int(tokens[i]))
                queue.append(node.right)
            i += 1
        return root


for vals in tests:
    assert run_codec_leetcode(CodecBFS, vals) == vals
    leetcode_text = "[" + ",".join("null" if v is None else str(v) for v in vals) + "]"
    assert CodecBFS().serialize(build_tree(vals)) == leetcode_text
print("CodecBFS PASS toàn bộ test; chuỗi trùng định dạng LeetCode, ví dụ:", CodecBFS().serialize(build_tree(ex1)))

CodecBFS PASS toàn bộ test; chuỗi trùng định dạng LeetCode, ví dụ: [1,2,3,null,null,4,5]


## 10. Cách 4 (tùy chọn) — Post-order đọc ngược, và pre-order + in-order

### 10.1 Post-order có `#`, đọc từ cuối về

Đổi thứ tự ghi thành post-order (trái → phải → node), `None` vẫn ghi `#`. Example 1 cho `"#,#,2,#,#,4,#,#,5,3,1"`: root `1` nằm ở **cuối chuỗi**. Đọc từ trái sang phải thì tới tận token cuối mới biết root là gì — bất tiện.

Mẹo: đọc **từ cuối về đầu**. Đảo ngược thứ tự "trái → phải → node" ta được "**node → phải → trái**": đó là pre-order "soi gương" (đi phải trước). Vậy deserialize:

1. lấy token **cuối** (`tokens.pop()` — xóa ở cuối list chỉ `O(1)`);
2. `#` → `None`; ngược lại tạo node;
3. dựng **con phải trước**, rồi mới tới con trái.

Kiểm tra bằng mắt: đảo ngược chuỗi trên được `1,3,5,#,#,4,#,#,2,#,#` — node `1`, rồi cả cây con **phải** `3,5,#,#,4,#,#`, rồi cây con **trái** `2,#,#`. Nếu lỡ dựng con trái trước, ta được cây **bị lật gương** (mirror) — cell dưới cho thấy lỗi này.

Độ phức tạp giống hệt cách 2: `O(n)` thời gian (mỗi `pop()` ở cuối list là `O(1)`), `O(h)` call stack, chuỗi dài đúng `2n + 1` token.

In [25]:
def post_serialize(root):
    tokens = []

    def walk(node):
        if node is None:
            tokens.append("#")
            return
        walk(node.left)
        walk(node.right)
        tokens.append(str(node.val))     # node ghi SAU CÙNG

    walk(root)
    return ",".join(tokens)


def post_deserialize(data):
    tokens = data.split(",")

    def build():
        tok = tokens.pop()               # lấy từ CUỐI
        if tok == "#":
            return None
        node = TreeNode(int(tok))
        node.right = build()             # phải TRƯỚC
        node.left = build()
        return node

    return build()


def post_deserialize_wrong(data):        # LỖI: dựng trái trước
    tokens = data.split(",")

    def build():
        tok = tokens.pop()
        if tok == "#":
            return None
        node = TreeNode(int(tok))
        node.left = build()
        node.right = build()
        return node

    return build()


s = post_serialize(root1)
print("post-order       :", s)
print("đảo ngược        :", ",".join(reversed(s.split(","))))
print("dựng đúng        :", tree_to_list(post_deserialize(s)), same_tree(post_deserialize(s), root1))
print("dựng trái trước  :", tree_to_list(post_deserialize_wrong(s)), same_tree(post_deserialize_wrong(s), root1))

post-order       : #,#,2,#,#,4,#,#,5,3,1
đảo ngược        : 1,3,5,#,#,4,#,#,2,#,#
dựng đúng        : [1, 2, 3, None, None, 4, 5] True
dựng trái trước  : [1, 3, 2, 5, 4] False


### 10.2 Pre-order + in-order (không cần `#`) — chỉ dùng được khi giá trị **đôi một khác nhau**

Mục 5 cho thấy **một** dãy duyệt không có `null` là không đủ. Nhưng **hai** dãy thì sao? Kết quả kinh điển (LeetCode 105): biết pre-order và in-order là dựng lại được cây — **với điều kiện các giá trị đôi một khác nhau**.

Cách dựng: `pre[0]` là root. Tìm vị trí `m` của root trong in-order: mọi thứ bên trái `m` thuộc cây con trái, bên phải `m` thuộc cây con phải. Đệ quy cho hai nửa, lấy các phần tử tiếp theo của pre-order bằng một iterator dùng chung. Serialize là `"pre|in"`, ví dụ Example 1: `"1,2,3,4,5|2,1,4,3,5"`. Nhờ dict `pos` (giá trị → vị trí trong in-order) tra trong `O(1)` trung bình, cả quá trình là `O(n)` thời gian, `O(n)` bộ nhớ.

**Vì sao cần giá trị khác nhau?** Bước "tìm vị trí root trong in-order" sẽ có **nhiều** vị trí nếu giá trị trùng, và không có cách nào biết chọn vị trí nào. Phản ví dụ nhỏ nhất: cây A `[1,1]` (con trái) và cây B `[1,null,1]` (con phải) có pre-order đều là `[1,1]`, in-order cũng đều là `[1,1]` → chuỗi `"1,1|1,1"` giống hệt nhau.

**Áp vào bài 297:** giá trị nằm trong `[-1000, 1000]` — chỉ có 2001 giá trị khác nhau, mà cây có thể tới `10^4` node, nên khi `n >= 2002` **chắc chắn** có giá trị trùng (nguyên lý chuồng bồ câu: nhét 2002 con chim vào 2001 chuồng thì có chuồng chứa 2 con). Vậy cách này **không dùng được** cho 297 — nhưng rất đáng biết cho bài 105/106, và nó cho thấy vai trò thật sự của `#`: `#` mang thông tin hình dạng, nên giá trị có trùng cũng chẳng sao.

In [26]:
def pre_in_serialize(root):
    return ",".join(map(str, preorder(root))) + "|" + ",".join(map(str, inorder(root)))


def pre_in_deserialize(data):
    left_part, right_part = data.split("|")
    if left_part == "":
        return None
    pre = [int(x) for x in left_part.split(",")]
    ino = [int(x) for x in right_part.split(",")]
    pos = {v: i for i, v in enumerate(ino)}       # CHỈ đúng khi giá trị khác nhau
    nxt = iter(pre)

    def build(lo, hi):                            # cây con ứng với ino[lo:hi]
        if lo >= hi:
            return None
        v = next(nxt)
        node = TreeNode(v)
        m = pos[v]
        node.left = build(lo, m)
        node.right = build(m + 1, hi)
        return node

    return build(0, len(ino))


s = pre_in_serialize(root1)
print(s, "->", tree_to_list(pre_in_deserialize(s)), same_tree(pre_in_deserialize(s), root1))
print("cây rỗng:", repr(pre_in_serialize(None)), "->", pre_in_deserialize(pre_in_serialize(None)))
print("Cây A [1,1]     :", pre_in_serialize(build_tree([1, 1])))
print("Cây B [1,null,1]:", pre_in_serialize(build_tree([1, None, 1])))
print("so với có '#'   :", dfs_serialize(build_tree([1, 1])), "vs", dfs_serialize(build_tree([1, None, 1])))

1,2,3,4,5|2,1,4,3,5 -> [1, 2, 3, None, None, 4, 5] True
cây rỗng: '|' -> None
Cây A [1,1]     : 1,1|1,1
Cây B [1,null,1]: 1,1|1,1
so với có '#'   : 1,1,#,#,# vs 1,#,1,#,#


## 11. Cây lệch sâu và giới hạn đệ quy

### 11.1 Vấn đề và ba cách xử lý

Constraints cho `n` tới `10^4`, nên cây lệch (mỗi node một con) sâu tới `10^4` tầng. `walk` / `build` của cách chính cần `10^4` lời gọi lồng nhau, vượt giới hạn mặc định (~1000) → `RecursionError`. Có ba cách xử lý:

1. **Dùng BFS** (mục 9): vốn chỉ là vòng lặp + queue, chạy ngon lành với mọi độ sâu.
2. **Tăng giới hạn đệ quy tạm thời**: `sys.setrecursionlimit(25_000)`, đặt trong `try ... finally` để **luôn trả lại** giá trị cũ dù có lỗi hay không (khối `finally` luôn chạy). Trên LeetCode, nhiều người đặt `sys.setrecursionlimit(...)` ở đầu file. Lưu ý: giới hạn này sinh ra để bảo vệ chương trình; trên các phiên bản Python cũ, đặt quá cao có thể làm chương trình **crash hẳn** (tràn stack thật của hệ điều hành) thay vì báo lỗi lịch sự.
3. **Tự viết DFS bằng stack** (mục 11.2): giữ nguyên định dạng `#` của cách chính nhưng không dùng đệ quy.

Cell dưới dựng cây lệch trái `10^4` node (đúng giới hạn đề), cho thấy cách chính vỡ ở giới hạn mặc định, rồi chạy được bằng cách 1 và cách 2.

In [27]:
deep = make_chain(10_000, "left")
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

try:
    Codec().serialize(deep)
except RecursionError as e:
    print("Codec (đệ quy) trên cây sâu 10^4 ->", type(e).__name__)

# Cách 1: BFS không đệ quy
s_bfs = CodecBFS().serialize(deep)
print("CodecBFS: độ dài chuỗi =", len(s_bfs), "| đúng cấu trúc:", same_tree_iter(deep, CodecBFS().deserialize(s_bfs)))

# Cách 2: tăng giới hạn TẠM THỜI rồi trả lại
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(25_000)
try:
    s_dfs = Codec().serialize(deep)
    back = Codec().deserialize(s_dfs)
finally:
    sys.setrecursionlimit(old_limit)
print("Codec với giới hạn 25000: số token =", len(s_dfs.split(",")), "| đúng cấu trúc:", same_tree_iter(deep, back))
print("giới hạn đã trả lại:", sys.getrecursionlimit())

giới hạn đệ quy mặc định: 1000
Codec (đệ quy) trên cây sâu 10^4 -> RecursionError
CodecBFS: độ dài chuỗi = 88891 | đúng cấu trúc: True
Codec với giới hạn 25000: số token = 20001 | đúng cấu trúc: True
giới hạn đã trả lại: 1000


### 11.2 DFS pre-order bằng stack tự quản lý (không đệ quy)

**Serialize** bằng stack: bắt đầu với `[root]`; mỗi lần `pop` một phần tử: `None` thì ghi `#`; node thật thì ghi giá trị rồi đẩy **con phải trước, con trái sau** — con trái nằm trên đỉnh nên được lấy ra trước, cho đúng thứ tự *node → trái → phải*.

**Deserialize** bằng một stack các **"chỗ trống đang chờ"**: mỗi phần tử là cặp `(cha, phía)` nghĩa là "ô con `phía` của node `cha` đang chờ token". Token đầu tiên là root; root mở hai chỗ trống, ta đẩy `(root, "phải")` rồi `(root, "trái")` (trái ở trên đỉnh). Với mỗi token tiếp theo:

- `pop` chỗ trống trên đỉnh `(cha, phía)`;
- token là `#` thì chỗ đó để `None`; token là số thì tạo node, gắn vào `cha` ở `phía`, và đẩy hai chỗ trống mới của nó: `(con, "phải")` rồi `(con, "trái")`.

Trace ngắn với Example 1 `1,2,#,#,3,4,#,#,5,#,#`:

| Token | Lấy chỗ trống | Hành động | Stack sau (đỉnh ở bên phải) |
|:---:|---|---|---|
| `1` | — | tạo root 1 | `(1,phải) (1,trái)` |
| `2` | `(1,trái)` | 1.left = 2 | `(1,phải) (2,phải) (2,trái)` |
| `#` | `(2,trái)` | None | `(1,phải) (2,phải)` |
| `#` | `(2,phải)` | None | `(1,phải)` |
| `3` | `(1,phải)` | 1.right = 3 | `(3,phải) (3,trái)` |
| `4`, `#`, `#` | ... | 3.left = 4, hai ô của 4 để None | `(3,phải)` |
| `5`, `#`, `#` | ... | 3.right = 5, hai ô của 5 để None | rỗng |

Số phần tử trên stack lúc nào cũng bằng **số chỗ trống đang mở** — ý tưởng "đếm chỗ trống" này chính là lời giải của LeetCode 331 (Bài 5 ở cuối notebook).

Độ phức tạp: vẫn `O(n)` thời gian; stack tự quản lý có thể dài tới `O(n)` (ví dụ cây lệch trái: mỗi node để lại một chỗ trống bên phải chờ tới cuối), nhưng nó nằm trong bộ nhớ thường nên không bị giới hạn đệ quy.

In [28]:
def dfs_serialize_iter(root):
    tokens, stack = [], [root]
    while stack:
        node = stack.pop()
        if node is None:
            tokens.append("#")
            continue
        tokens.append(str(node.val))
        stack.append(node.right)         # phải vào trước -> ra sau
        stack.append(node.left)          # trái vào sau  -> ra trước
    return ",".join(tokens)


def dfs_deserialize_iter(data):
    tokens = data.split(",")
    if tokens[0] == "#":
        return None
    root = TreeNode(int(tokens[0]))
    slots = [(root, "right"), (root, "left")]    # các chỗ trống đang chờ, đỉnh ở cuối list
    for tok in tokens[1:]:
        parent, side = slots.pop()
        if tok == "#":
            continue                             # chỗ trống này giữ None
        child = TreeNode(int(tok))
        if side == "left":
            parent.left = child
        else:
            parent.right = child
        slots.append((child, "right"))
        slots.append((child, "left"))
    return root


print(dfs_serialize_iter(root1), "| trùng bản đệ quy:", dfs_serialize_iter(root1) == dfs_serialize(root1))
print("dựng lại:", tree_to_list(dfs_deserialize_iter("1,2,#,#,3,4,#,#,5,#,#")))
for side in ["left", "right", "zigzag"]:
    chain = make_chain(10_000, side)
    s = dfs_serialize_iter(chain)
    print(f"cây lệch {side:<6} 10^4 node: {len(s.split(','))} token, đúng cấu trúc:",
          same_tree_iter(chain, dfs_deserialize_iter(s)))

1,2,#,#,3,4,#,#,5,#,# | trùng bản đệ quy: True
dựng lại: [1, 2, 3, None, None, 4, 5]
cây lệch left   10^4 node: 20001 token, đúng cấu trúc: True
cây lệch right  10^4 node: 20001 token, đúng cấu trúc: True


cây lệch zigzag 10^4 node: 20001 token, đúng cấu trúc: True


## 12. Đối chiếu chéo mọi cách trên cây ngẫu nhiên có `seed`

Tới đây ta có nhiều cặp serialize/deserialize. Kiểm tra tất cả cùng lúc bằng **round-trip**: với mỗi cây `T`, kiểm tra `same_tree_iter(deserialize(serialize(T)), T)`.

**Sinh cây ngẫu nhiên** `random_tree(n, lo, hi, distinct)`: bắt đầu từ root; giữ một danh sách các **chỗ trống** (cặp `(cha, phía)`) có thể gắn node mới; mỗi bước chọn ngẫu nhiên một chỗ trống, gắn node mới vào đó, xóa chỗ trống đó khỏi danh sách và thêm hai chỗ trống của node mới. Cách này sinh đủ loại hình dạng: cân bằng, lệch, răng cưa... Mẹo xóa nhanh `O(1)` khỏi list: chép phần tử cuối đè lên vị trí cần xóa rồi `pop()` phần tử cuối (thứ tự trong danh sách chỗ trống không quan trọng). Với `distinct=True`, các giá trị được rút bằng `random.sample` nên đôi một khác nhau (dùng cho cách pre + in).

Ba nhóm test:

1. **Ngẫu nhiên nhỏ** (`random.seed(297)`): 800 cây, `n` từ 0 đến 30; một nửa giá trị trong `[-3, 3]` (rất nhiều số **trùng**, số **0**, số âm), một nửa trong `[-1000, 1000]`. Chạy với **mọi** cách; riêng pre + in test trên 300 cây có giá trị khác nhau.
2. **Vét cạn hình dạng**: mọi hình dạng `n <= 7` (626 cây), tất cả giá trị bằng `0` — trường hợp khó nhất vì **chỉ còn hình dạng** để phân biệt. Kiểm tra mọi cách round-trip đúng, chuỗi có `#` **khác nhau với mọi hình dạng**, còn pre-order không `null` thì chỉ có **một** chuỗi cho mỗi `n`.
3. **Cây lớn đúng giới hạn đề**: một cây ngẫu nhiên `10^4` node và 3 cây lệch `10^4` node, với các cách không đệ quy.

Trong danh sách `methods`, `Codec().serialize` là method gắn với một object (đóng vai `ser`), còn `Codec().deserialize` gắn với một object **khác** (đóng vai `deser`) — dùng được y như hàm bình thường, và đúng tinh thần hai object của LeetCode.

In [29]:
def random_tree(n, lo=-1000, hi=1000, distinct=False):
    if n == 0:
        return None
    if distinct:
        values = random.sample(range(lo, hi + 1), n)       # n giá trị đôi một khác nhau
    else:
        values = [random.randint(lo, hi) for _ in range(n)]
    root = TreeNode(values[0])
    slots = [(root, "left"), (root, "right")]              # các chỗ trống có thể gắn node mới
    for v in values[1:]:
        k = random.randrange(len(slots))
        parent, side = slots[k]
        slots[k] = slots[-1]                               # xóa slots[k] trong O(1)
        slots.pop()
        child = TreeNode(v)
        if side == "left":
            parent.left = child
        else:
            parent.right = child
        slots.append((child, "left"))
        slots.append((child, "right"))
    return root


methods = [
    ("JSON",                 json_serialize,     json_deserialize),
    ("tuple + literal_eval", tuple_serialize,    tuple_deserialize),
    ("DFS # (đệ quy)",       dfs_serialize,      dfs_deserialize),
    ("DFS # (stack)",        dfs_serialize_iter, dfs_deserialize_iter),
    ("BFS",                  bfs_serialize,      bfs_deserialize),
    ("post-order đọc ngược", post_serialize,     post_deserialize),
    ("Codec",                Codec().serialize,    Codec().deserialize),
    ("CodecBFS",             CodecBFS().serialize, CodecBFS().deserialize),
]

# Nhóm 1: ngẫu nhiên nhỏ
random.seed(297)
trees = [random_tree(random.randint(0, 30), -3, 3) for _ in range(400)]
trees += [random_tree(random.randint(0, 30)) for _ in range(400)]
for name, ser_fn, deser_fn in methods:
    assert all(same_tree_iter(deser_fn(ser_fn(t)), t) for t in trees), name
print(f"Nhóm 1: {len(methods)} cách x {len(trees)} cây ngẫu nhiên PASS")
distinct_trees = [random_tree(random.randint(0, 30), distinct=True) for _ in range(300)]
assert all(same_tree_iter(pre_in_deserialize(pre_in_serialize(t)), t) for t in distinct_trees)
print("        pre + in trên 300 cây giá trị khác nhau PASS")

# Nhóm 2: vét cạn hình dạng, mọi giá trị = 0
print("Nhóm 2:  n | số hình dạng | chuỗi có '#' khác nhau | pre-order không null khác nhau")
for n in range(8):
    zero_trees = [shape_to_tree(sh, iter([0] * n)) for sh in all_shapes(n)]
    for name, ser_fn, deser_fn in methods:
        assert all(same_tree_iter(deser_fn(ser_fn(t)), t) for t in zero_trees), name
    with_hash = {dfs_serialize(t) for t in zero_trees}
    no_null = {preorder_no_null(t) for t in zero_trees}
    assert len(with_hash) == len(zero_trees)
    print(f"        {n} | {len(zero_trees):>12} | {len(with_hash):>22} | {len(no_null):>3}")

# Nhóm 3: cây lớn đúng giới hạn đề, các cách không đệ quy
big_trees = [random_tree(10_000)] + [make_chain(10_000, side) for side in ("left", "right", "zigzag")]
for name, ser_fn, deser_fn in methods:
    if name in ("DFS # (stack)", "BFS", "CodecBFS"):
        assert all(same_tree_iter(deser_fn(ser_fn(t)), t) for t in big_trees), name
        print(f"Nhóm 3: {name:<14} PASS trên 4 cây 10^4 node (1 ngẫu nhiên + 3 lệch)")

Nhóm 1: 8 cách x 800 cây ngẫu nhiên PASS
        pre + in trên 300 cây giá trị khác nhau PASS
Nhóm 2:  n | số hình dạng | chuỗi có '#' khác nhau | pre-order không null khác nhau
        0 |            1 |                      1 |   1
        1 |            1 |                      1 |   1
        2 |            2 |                      2 |   1
        3 |            5 |                      5 |   1
        4 |           14 |                     14 |   1
        5 |           42 |                     42 |   1


        6 |          132 |                    132 |   1
        7 |          429 |                    429 |   1
Nhóm 3: DFS # (stack)  PASS trên 4 cây 10^4 node (1 ngẫu nhiên + 3 lệch)
Nhóm 3: BFS            PASS trên 4 cây 10^4 node (1 ngẫu nhiên + 3 lệch)
Nhóm 3: CodecBFS       PASS trên 4 cây 10^4 node (1 ngẫu nhiên + 3 lệch)


## 13. So sánh độ dài chuỗi giữa các cách (và việc cắt `null` thừa ở cuối)

Đề 297 không bắt chuỗi phải ngắn nhất (bài anh em 449 thì có), nhưng độ dài chuỗi = dung lượng file / băng thông mạng, nên đáng để so. Trước khi chạy, hãy đếm **token**:

| Cách | Số token | Vì sao |
|---|---|---|
| DFS `#`, post-order | luôn `2n + 1` | `n` giá trị + `n + 1` chỗ trống (mục 5.3) |
| BFS chưa cắt | luôn `2n + 1` | cũng ghi đủ `n` giá trị + `n + 1` chỗ trống, chỉ khác thứ tự |
| BFS đã cắt | `2n + 1 - (số null ở cuối)` | luôn cắt được ít nhất 2 (hai con của node cuối cùng theo tầng — chắc chắn là lá) |
| pre + in | `2n` | mỗi giá trị ghi **hai lần**, không có chỗ trống |

Còn về **ký tự**: `"null"` dài 4 ký tự, `#` chỉ 1, nên ở bảng dưới có thêm cột "BFS đã cắt, dùng `#`" để so công bằng với DFS. Năm cây thử: Example 1; một cây ngẫu nhiên 150 node (giá trị trong `[-1000, 1000]`, nên mỗi giá trị dài 1–5 ký tự); cây **đầy đủ** (perfect — mọi tầng đều kín) 127 node; cây lệch trái và lệch phải 150 node. (Chọn 150 tầng để cả JSON và tuple + `literal_eval` không bị vỡ.)

Cú pháp mới trong cell: `lambda r: bfs_serialize(r, trim=False)` là cách viết tắt một **hàm một dòng** — nhận `r` và trả về biểu thức sau dấu `:`. Nó tiện khi cần "gói" một lời gọi có tham số phụ thành một hàm chỉ nhận cây.

In [30]:
random.seed(13)
sample_trees = [
    ("Ex1 n=5",          root1),
    ("ngẫu nhiên n=150", random_tree(150)),
    ("đầy đủ n=127",     build_tree(list(range(1, 128)))),
    ("lệch trái n=150",  make_chain(150, "left")),
    ("lệch phải n=150",  make_chain(150, "right")),
]
length_methods = [
    ("JSON",                lambda r: json_serialize(r)),
    ("tuple repr",          lambda r: tuple_serialize(r)),
    ("DFS #",               lambda r: dfs_serialize(r)),
    ("post-order #",        lambda r: post_serialize(r)),
    ("BFS chưa cắt",        lambda r: bfs_serialize(r, trim=False)),
    ("BFS đã cắt (LeetCode)", lambda r: bfs_serialize(r)),
    ("BFS đã cắt, dùng #",  lambda r: bfs_serialize(r).replace("null", "#")),
    ("pre + in",            lambda r: pre_in_serialize(r)),
]

print(f"{'số KÝ TỰ':<22}" + "".join(f"{name:>18}" for name, _ in sample_trees))
for mname, fn in length_methods:
    print(f"{mname:<22}" + "".join(f"{len(fn(t)):>18}" for _, t in sample_trees))

print()
print(f"{'số TOKEN':<22}" + "".join(f"{name:>18}" for name, _ in sample_trees))
print(f"{'DFS # (= 2n+1)':<22}" + "".join(f"{len(dfs_serialize(t).split(',')):>18}" for _, t in sample_trees))
print(f"{'BFS đã cắt':<22}" + "".join(f"{len(bfs_serialize(t)[1:-1].split(',')):>18}" for _, t in sample_trees))

số KÝ TỰ                         Ex1 n=5  ngẫu nhiên n=150      đầy đủ n=127   lệch trái n=150   lệch phải n=150
JSON                                 179              5609              4595              5444              5444
tuple repr                            59              2009              1547              1844              1844
DFS #                                 21               956               655               791               791
post-order #                          21               956               655               791               791
BFS chưa cắt                          41              1411              1041              1246              1246
BFS đã cắt (LeetCode)                 21              1311               401              1231              1236
BFS đã cắt, dùng #                    15               918               401               787               789
pre + in                              19              1309               799               979  

Đọc bảng:

- **JSON dài nhất** — gấp khoảng 6–9 lần DFS `#` (Example 1: 179 so với 21 ký tự), vì lặp tên khóa ở mọi node. `tuple repr` đỡ hơn nhưng vẫn gấp khoảng 2–3 lần (mỗi `None` tốn 4 ký tự, cộng thêm ngoặc và dấu cách).
- **DFS `#` và post-order `#` dài bằng nhau** trên mọi cây: cùng các token, chỉ khác thứ tự.
- **BFS chưa cắt** có cùng **số token** `2n + 1` như DFS, nhưng dài hơn hẳn về ký tự vì `null` dài 4 ký tự.
- **Cắt `null` cuối** hiệu quả nhất với cây **đầy đủ**: cả tầng chỗ trống cuối cùng (128 cái) nằm liền ở đuôi chuỗi và bị cắt sạch, chỉ còn **127 token = n** (DFS vẫn là 255). Với cây lệch thì chỉ cắt được 2–3 `null`, còn gần `2n` token: lệch phải còn `2n - 1 = 299` (cắt hai con của lá cuối), lệch trái còn `2n - 2 = 298` (cắt thêm được con phải rỗng của node áp cuối, vì nó cũng rơi vào đuôi chuỗi).
- **BFS đã cắt, dùng `#`** không bao giờ dài hơn DFS `#` — cùng tập token, lại bỏ bớt được đuôi.
- **pre + in** ngắn với cây ít chữ số (Example 1), nhưng với giá trị nhiều chữ số thì ghi mỗi giá trị hai lần lại tốn hơn ghi thêm `n + 1` dấu `#` một ký tự — và nó còn **sai** khi giá trị trùng.

Kết luận thực tế: mọi cách "tự thiết kế" đều `O(n)` ký tự; chênh lệch chỉ là hằng số. Chọn cách dễ viết đúng nhất (DFS `#`) hoặc cách khớp định dạng cần dùng (BFS).

## 14. Vì sao cách chính đúng? Độ phức tạp và so sánh các cách

### 14.1 Chứng minh cách DFS `#` (quy nạp theo số node)

Gọi `ser(T)` là dãy token của cây `T`. **Khẳng định:** với mọi cây `T`, nếu băng chuyền đang đứng ở **đầu** đoạn `ser(T)` (phía sau còn gì cũng được), thì một lời gọi `build()` sẽ trả về một cây **giống hệt `T`** và dừng **ngay sau token cuối** của `ser(T)`.

- **`T` rỗng:** `ser(T) = #`. `build()` đọc `#`, trả `None`, đã đọc đúng 1 token. ✓
- **`T` có root `v`, cây con trái `L`, cây con phải `R`:** `ser(T) = v, ser(L), ser(R)`. `build()` đọc `v` và tạo node. Lúc này băng chuyền đứng ở đầu `ser(L)`; vì `L` ít node hơn `T`, theo giả thiết quy nạp, lời gọi `build()` thứ nhất trả về đúng `L` và dừng ngay sau `ser(L)` — tức là ở **đầu `ser(R)`**. Lời gọi thứ hai, tương tự, trả về đúng `R` và dừng ngay sau `ser(R)`, cũng là cuối `ser(T)`. ✓

**Hệ quả 1:** `deserialize(serialize(T))` dựng lại đúng `T`, và đọc **hết** chuỗi, không thừa không thiếu token (trace 7.2 in "token còn thừa: []").

**Hệ quả 2 — mỗi cây một chuỗi riêng:** nếu hai cây khác nhau `A ≠ B` cho cùng một chuỗi `s`, thì `deserialize(s)` phải vừa bằng `A` vừa bằng `B` — vô lý. Đây đúng là điều pre-order không `#` **không** có được (mục 5.2: 5 cây, 1 chuỗi; không thể tồn tại hàm giải mã).

### 14.2 Vì sao cách BFS đúng

Serialize và deserialize cùng lấy các node thật ra khỏi queue theo **cùng một thứ tự** (queue FIFO, cùng quy tắc "xếp con trái trước con phải"). Node thật thứ `k` ghi hai con của nó ở vị trí `2k - 1` và `2k`, và deserialize cũng đọc đúng hai vị trí đó cho node thứ `k` (Example 1: node `1` → vị trí 1, 2; node `2` → 3, 4; node `3` → 5, 6; node `4` → 7, 8 đã bị cắt nên giữ `None`). Token bị cắt ở cuối đều là `null`, và "hết token" được hiểu đúng là `None`.

### 14.3 Độ phức tạp

Ký hiệu: `n` = số node, `h` = chiều cao, `w` = số node nhiều nhất trên một tầng (`w <= n`). Mỗi giá trị dài tối đa 5 ký tự (`-1000`), nên độ dài chuỗi là `O(n)` ký tự và `int(tok)` coi như `O(1)`.

- **Thời gian `O(n)`** cho cả serialize và deserialize ở mọi cách: mỗi node được thăm một lần, mỗi token được đọc một lần; `split` và `",".join` tuyến tính theo độ dài chuỗi.
- **Bộ nhớ:** chuỗi kết quả và list token là `O(n)` (không tránh được — đầu ra đã dài cỡ đó). Phần **phụ** khác nhau: call stack `O(h)` cho bản đệ quy; queue `O(w)` cho BFS; stack tự quản lý `O(n)` trong trường hợp xấu cho bản lặp.

| Cách | Time | Bộ nhớ phụ | Độ dài chuỗi | Cây sâu `10^4` | Ghi chú |
|---|---|---|---|---|---|
| 1. JSON lồng nhau | `O(n)` | `O(n)` + `O(h)` đệ quy | dài nhất | vỡ | dễ viết, phụ thuộc thư viện |
| 1b. tuple + `literal_eval` | `O(n)` | `O(n)` + `O(h)` | dài | vỡ từ ~200 tầng | đừng bao giờ dùng `eval` |
| **2. DFS `#` (đệ quy)** | `O(n)` | `O(h)` | `2n + 1` token | cần tăng giới hạn | **cách chính**, dễ chứng minh |
| 2b. DFS `#` (stack) | `O(n)` | `O(n)` | `2n + 1` token | OK | cùng chuỗi với bản đệ quy |
| **3. BFS** | `O(n)` | `O(w)` | `<= 2n - 1` token sau khi cắt | OK | khớp định dạng LeetCode |
| 4a. post-order đọc ngược | `O(n)` | `O(h)` | `2n + 1` token | cần tăng giới hạn | nhớ dựng **phải trước** |
| 4b. pre + in | `O(n)` (dùng dict) | `O(n)` | `2n` token | cần tăng giới hạn | **sai khi giá trị trùng** → không dùng cho 297 |

## 15. Lỗi thường gặp

### Lỗi 1: "Giấu" cây trong `self` thay vì ghi vào chuỗi

```python
class CheatCodec:
    def serialize(self, root):
        self.saved = root         # cất cây trong object ser
        return "đã lưu"
    def deserialize(self, data):
        return self.saved         # nhưng deser là object KHÁC -> AttributeError
```

LeetCode dùng hai object `ser` và `deser` khác nhau, nên `deser` không có `saved`. Kể cả khi "lách" được bằng biến toàn cục, đó **không phải** serialize — chuỗi `"đã lưu"` không thể gửi sang máy khác mà dựng lại cây. **Sửa:** mọi thông tin phải nằm trong chuỗi.

### Lỗi 2: Không ghi chỗ trống

Pre-order (hay bất kỳ thứ tự duyệt nào) chỉ ghi giá trị thì nhiều cây cho cùng một chuỗi (mục 5). **Sửa:** ghi `#` (hoặc `null`) cho mọi `None`.

### Lỗi 3: Không có dấu phân cách giữa các token

Ghi liền `"123###"` thì không biết đó là `12` có con trái `3`, hay `1` có con trái `23`. **Sửa:** luôn có dấu phân cách (`,`) và tách bằng `split(",")`, **không** duyệt từng ký tự.

### Lỗi 4: Kiểm tra kiểu "truthy" làm mất giá trị `0`

```python
if vals[i]:          # SAI: 0 cũng bị coi như "không có node"
if vals[i] is not None:   # ĐÚNG (hoặc so sánh token với "null"/"#")
```

### Lỗi 5: Cây rỗng

Nếu serialize cây rỗng thành `""`, thì `"".split(",")` là `['']` và `int('')` báo `ValueError`. **Sửa:** chọn một chuỗi rõ ràng cho cây rỗng (`"#"` hoặc `"[]"`) và test nó; hoặc kiểm tra chuỗi rỗng trước khi đọc.

### Lỗi 6: Gọi `TreeNode(val, left, right)`

`TreeNode` của bài 297 chỉ nhận `x` → `TypeError`. **Sửa:** `node = TreeNode(int(tok))` rồi gán `node.left`, `node.right`.

### Lỗi 7: Dùng `tokens.pop(0)` trong deserialize

Mỗi lần `pop(0)` phải dời **mọi** phần tử còn lại lên một ô: `O(n)` mỗi lần, `O(n²)` tổng cộng. **Sửa:** `iter` + `next`, `deque.popleft()`, một chỉ số `i`, hoặc đảo list rồi `pop()` từ cuối.

### Lỗi 8: Cộng chuỗi và dư dấu phẩy

`s += str(node.val) + ","` sinh ra `"1,2,#,"` — dấu phẩy cuối làm `split` trả thêm token rỗng `''` ở cuối (với DFS thì token đó không bao giờ được đọc nên "tình cờ" vẫn chạy, nhưng với cách khác sẽ vỡ), và cộng chuỗi liên tục có thể chậm. **Sửa:** gom list rồi `",".join(...)`.

### Lỗi 9: Post-order đọc ngược mà dựng con trái trước → cây bị lật gương (đã demo ở mục 10.1).

### Lỗi 10: Quên giới hạn đệ quy với cây lệch sâu (mục 11).

Cell dưới tái hiện các lỗi 1, 3, 4, 5, 6 và đo lỗi 7.

In [31]:
# Lỗi 1: giấu cây trong self
class CheatCodec:
    def serialize(self, root):
        self.saved = root
        return "đã lưu"

    def deserialize(self, data):
        return self.saved


try:
    run_codec_leetcode(CheatCodec, ex1)
except AttributeError as e:
    print("Lỗi 1 ->", type(e).__name__, ":", e)

# Lỗi 3: không có dấu phân cách
tree_12_3 = build_tree([12, 3])          # 12 có con trái 3
tree_1_23 = build_tree([1, 23])          # 1 có con trái 23
print("Lỗi 3 -> không dấu phẩy:", dfs_serialize(tree_12_3).replace(",", ""), "vs",
      dfs_serialize(tree_1_23).replace(",", ""), "| có dấu phẩy:", dfs_serialize(tree_12_3), "vs", dfs_serialize(tree_1_23))


# Lỗi 4: kiểm tra truthy
def bfs_deserialize_truthy_bug(data):
    vals = [None if t == "null" else int(t) for t in data[1:-1].split(",")]
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        if vals[i]:                                   # BUG
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i]:                 # BUG
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root


print("Lỗi 4 ->", tree_to_list(bfs_deserialize_truthy_bug("[0,0,0]")), "thay vì", tree_to_list(bfs_deserialize("[0,0,0]")))


# Lỗi 5: cây rỗng thành "" rồi đọc thẳng tokens[0]
def deserialize_no_empty_check(data):
    tokens = data.split(",")                          # "" -> ['']
    return TreeNode(int(tokens[0]))


try:
    deserialize_no_empty_check("")
except ValueError as e:
    print("Lỗi 5 ->", type(e).__name__, ":", e)

# Lỗi 6: TreeNode của bài 297 chỉ nhận một tham số
try:
    TreeNode(1, None, None)
except TypeError as e:
    print("Lỗi 6 ->", type(e).__name__, ":", e)


# Lỗi 7: pop(0) là O(n^2) — gấp đôi n thì thời gian gấp khoảng 4
def time_consume(n, use_pop0):
    data = ["1"] * n
    start = time.perf_counter()
    if use_pop0:
        while data:
            data.pop(0)
    else:
        it = iter(data)
        for _ in range(n):
            next(it)
    return time.perf_counter() - start


for n in [20_000, 40_000, 80_000]:
    print(f"Lỗi 7 -> n = {n:>6}: pop(0) {time_consume(n, True) * 1000:7.1f} ms | iter/next {time_consume(n, False) * 1000:5.1f} ms")

Lỗi 1 -> AttributeError : 'CheatCodec' object has no attribute 'saved'
Lỗi 3 -> không dấu phẩy: 123### vs 123### | có dấu phẩy: 12,3,#,#,# vs 1,23,#,#,#
Lỗi 4 -> [0] thay vì [0, 0, 0]
Lỗi 5 -> ValueError : invalid literal for int() with base 10: ''
Lỗi 6 -> TypeError : TreeNode.__init__() takes 2 positional arguments but 4 were given


Lỗi 7 -> n =  20000: pop(0)    30.3 ms | iter/next   0.5 ms
Lỗi 7 -> n =  40000: pop(0)   135.3 ms | iter/next   1.0 ms


Lỗi 7 -> n =  80000: pop(0)   535.5 ms | iter/next   2.0 ms


## 16. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nên nghĩ tới serialize

- Đề nói **lưu / gửi / khôi phục** một cấu trúc (cây, đồ thị, object) → thiết kế định dạng chuỗi + hàm đọc ngược.
- Cần **so sánh** cây hoặc **tìm cây con trùng nhau** → biến mỗi cây con thành một chuỗi "chữ ký" rồi dùng làm key của `dict` / `set`.
- Cần **dựng cây từ một dãy duyệt** → pre-order + một con trỏ đọc dùng chung (iterator). Nếu dãy **không có** ký hiệu `null`, phải có thêm thông tin: dãy in-order (giá trị khác nhau) hoặc tính chất của BST.
- Cần **kiểm tra một chuỗi pre-order có hợp lệ** không → đếm số chỗ trống đang mở.

### Khung tư duy nhớ nhanh

```text
serialize  : ghi node -> ghi trái -> ghi phải ; None ghi "#"
deserialize: đọc 1 token ; "#" -> None ; số -> tạo node, đọc trái, đọc phải   (một iterator dùng chung)
```

### Bài liên quan

| Bài | Liên hệ với 297 |
|---|---|
| 449. Serialize and Deserialize BST (Medium) | BST: pre-order **không cần** `#`, vì thứ tự giá trị tự cho biết đâu là ranh giới trái/phải (Bài 6) |
| 331. Verify Preorder Serialization of a Binary Tree (Medium) | đúng định dạng `#` của mục 6; đếm chỗ trống (Bài 5) |
| 428. Serialize and Deserialize N-ary Tree (Hard, Premium) | mỗi node có nhiều con → ghi thêm **số con** của mỗi node |
| 652. Find Duplicate Subtrees (Medium) | serialize **từng cây con** làm chữ ký để đếm trùng (demo dưới) |
| 572. Subtree of Another Tree (Easy) | serialize cả hai cây có `#`, kiểm tra chuỗi con — cẩn thận ranh giới token (`"2"` nằm trong `"12"`) |
| 105 / 106. Construct Binary Tree from Preorder (Postorder) and Inorder Traversal (Medium) | đúng mục 10.2 — cần giá trị khác nhau |
| 1028. Recover a Tree From Preorder Traversal (Hard) | pre-order kèm **độ sâu** (số dấu `-`) thay cho `#` |

Demo nhỏ cho 652: duyệt post-order, chữ ký của mỗi cây con là `"val,chữ_ký_trái,chữ_ký_phải"` (với `None` là `#`) — chính là chuỗi DFS `#` của cây con đó. Chữ ký nào xuất hiện **lần thứ hai** thì ghi nhận cây con đó. Với Example 1 của 652, `root = [1,2,3,4,null,2,4,null,null,4]`, đáp án là `[[2,4],[4]]`. (Bản này nối chuỗi nên có thể tốn `O(n²)` với cây sâu; bản tối ưu đổi mỗi chữ ký thành một số nguyên id.)

In [32]:
def find_duplicate_subtrees(root):
    seen = {}                        # chữ ký -> số lần gặp
    result = []

    def sig(node):
        if node is None:
            return "#"
        s = f"{node.val},{sig(node.left)},{sig(node.right)}"
        seen[s] = seen.get(s, 0) + 1
        if seen[s] == 2:             # chỉ ghi nhận đúng một lần cho mỗi chữ ký
            result.append(node)
        return s

    sig(root)
    return result


r652 = build_tree([1, 2, 3, 4, None, 2, 4, None, None, 4])
print_tree(r652)
dups = sorted(tree_to_list(t) for t in find_duplicate_subtrees(r652))
print("cây con trùng:", dups)
assert dups == [[2, 4], [4]]

1
├── L: 2
│   ├── L: 4
│   └── R: ·
└── R: 3
    ├── L: 2
    │   ├── L: 4
    │   └── R: ·
    └── R: 4
cây con trùng: [[2, 4], [4]]


## 17. Bài tập vận dụng

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý ở mục 18, đáp án ở mục 19 (cuối notebook) — hãy tự làm trước!

### Bài 1 — Đoán output (không chạy code)

```python
t = build_tree([5, 3, 8, None, 4])
print(dfs_serialize(t))
print(bfs_serialize(t, trim=False))
print(bfs_serialize(t))
print(dfs_serialize(t).count("#"))
```

Viết ra 4 dòng output. Dòng cuối có thể đoán mà **không cần** viết chuỗi ra không? Vì sao?

### Bài 2 — Tính tay bằng bảng

Cho chuỗi DFS `"1,2,#,4,#,#,3,#,5,6,#,#,#"`. Hãy:

1. lập bảng giống mục 7.2 (Bước, Token, Vị trí đang dựng, Hành động) và vẽ cây;
2. viết cây đó theo định dạng LeetCode (BFS đã cắt `null`);
3. viết chuỗi post-order có `#` của nó.

### Bài 3 — Điền TODO: lời giải chính

Hoàn thiện `CodecExercise` theo cách DFS pre-order có `#`, dùng `iter` + `next`.

In [33]:
class CodecExercise:

    def serialize(self, root):
        # TODO 1: tạo list tokens
        # TODO 2: viết hàm con walk(node): None -> ghi "#"; ngược lại ghi str(node.val), rồi walk trái, walk phải
        # TODO 3: gọi walk(root) rồi trả về ",".join(tokens)
        pass

    def deserialize(self, data):
        # TODO 4: tạo băng chuyền tokens = iter(data.split(","))
        # TODO 5: viết build(): tok = next(tokens); "#" -> None; số -> TreeNode(int(tok)), dựng trái rồi phải
        # TODO 6: trả về build()
        pass


# Tự làm xong rồi mới mở test:
# for vals in tests:
#     assert run_codec_leetcode(CodecExercise, vals) == vals
# assert CodecExercise().serialize(build_tree([1, 2])) != CodecExercise().serialize(build_tree([1, None, 2]))

### Bài 4 — Tự viết lại: deserialize định dạng LeetCode bằng queue

Viết `bfs_deserialize_exercise(data)` nhận chuỗi kiểu `"[1,2,3,null,null,4,5]"` (đã cắt `null` cuối, cây rỗng là `"[]"`) và trả về root. Không nhìn lại mục 9.3 nếu có thể. Hai chỗ hay sai: quên gỡ cặp ngoặc, và quên kiểm tra `i` còn trong phạm vi trước khi đọc con phải.

In [34]:
def bfs_deserialize_exercise(data):
    # TODO 1: gỡ "[" và "]"; nếu phần còn lại rỗng thì trả None
    # TODO 2: tokens = ...split(","); root từ tokens[0]; queue = deque([root]); i = 1
    # TODO 3: while queue and i < len(tokens): lấy node đầu queue,
    #         gắn con trái từ tokens[i], con phải từ tokens[i + 1] (bỏ qua "null", xếp con thật vào queue)
    raise NotImplementedError


# Gợi ý test:
# for vals in tests:
#     assert tree_to_list(bfs_deserialize_exercise(bfs_serialize(build_tree(vals)))) == vals
# assert bfs_deserialize_exercise("[]") is None

### Bài 5 — Biến thể: LeetCode 331. Verify Preorder Serialization of a Binary Tree

Đề 331 dùng **đúng** định dạng của cách chính: pre-order, `#` cho node rỗng, ngăn cách bởi dấu phẩy. Cho chuỗi `preorder`, trả về `True` nếu nó là chuỗi serialize hợp lệ của **một** cây nhị phân. Đề **không cho phép dựng lại cây**. Constraints: `1 <= preorder.length <= 10^4`; `preorder` gồm các số nguyên trong khoảng `[0, 100]` và ký tự `#`, ngăn cách bởi dấu phẩy.

```text
Input: preorder = "9,3,4,#,#,1,#,#,2,#,6,#,#"   Output: true
Input: preorder = "1,#"                          Output: false
Input: preorder = "9,#,#,1"                      Output: false
```

Gợi ý nhỏ: nhớ lại stack "chỗ trống đang chờ" ở mục 11.2 — bạn chỉ cần **đếm** số chỗ trống, không cần nhớ chúng là của node nào.

In [35]:
class Solution:
    def isValidSerialization(self, preorder: str) -> bool:
        # TODO: slots = 1 (chỗ của root); mỗi token lấp 1 chỗ; node thật mở thêm 2 chỗ
        # TODO: nếu cần đặt token mà slots == 0 -> False; cuối cùng slots phải bằng 0
        pass


# Gợi ý test:
# s331 = Solution()
# assert s331.isValidSerialization("9,3,4,#,#,1,#,#,2,#,6,#,#") == True
# assert s331.isValidSerialization("1,#") == False
# assert s331.isValidSerialization("9,#,#,1") == False
# assert s331.isValidSerialization("#") == True

### Bài 6 — Mini challenge: LeetCode 449. Serialize and Deserialize BST — không cần `#`

**BST** (Binary Search Tree — cây nhị phân tìm kiếm): với mọi node, mọi giá trị trong cây con trái **nhỏ hơn** node, mọi giá trị trong cây con phải **lớn hơn** node. Đề 449 yêu cầu serialize/deserialize một BST với chuỗi **càng gọn càng tốt** (constraints: số node trong `[0, 10^4]`, `0 <= Node.val <= 10^4`, input chắc chắn là BST).

Thử thách: chỉ ghi **pre-order các giá trị, không có `#`**, mà vẫn dựng lại đúng. Điều này có mâu thuẫn với mục 5 không? Không — mục 5 nói về cây nhị phân **bất kỳ**; ở BST, thứ tự giá trị đã mang sẵn thông tin hình dạng.

```python
bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
serialize_bst(bst)   -> "8,3,1,6,4,7,10,14,13"
deserialize_bst("8,3,1,6,4,7,10,14,13")  -> dựng lại đúng cây bst
serialize_bst(None)  -> ""     và   deserialize_bst("") -> None
```

Gợi ý nhỏ: một lời gọi dựng cây con nhận **khoảng giá trị hợp lệ** `(lo, hi)`. Nếu token kế tiếp không nằm trong khoảng đó thì nó không thuộc cây con này → trả `None` **mà không tiêu thụ** token.

In [36]:
def serialize_bst(root):
    # TODO: pre-order các giá trị, KHÔNG ghi '#'; cây rỗng -> ""
    pass


def deserialize_bst(data):
    # TODO: build(lo, hi): nếu hết token hoặc token kế tiếp không thuộc (lo, hi) -> None (không tiêu thụ)
    #       ngược lại lấy token, tạo node v, node.left = build(lo, v), node.right = build(v, hi)
    pass


# Gợi ý test:
# bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
# assert serialize_bst(bst) == "8,3,1,6,4,7,10,14,13"
# assert same_tree(deserialize_bst(serialize_bst(bst)), bst)
# assert serialize_bst(None) == "" and deserialize_bst("") is None

## 18. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Vẽ cây trước: `5` có con trái `3`, con phải `8`; `3` chỉ có con **phải** `4`. | DFS: node, trái, phải; BFS: từng tầng, `None` cũng ghi. | Số `#` = số chỗ trống = `n + 1`. |
| 2 | Đọc từng token, luôn biết mình đang dựng "trái của ai" hay "phải của ai". | Sau `#` của con trái thì chuyển sang con phải của **cùng** node; sau `#` của con phải thì node đó **xong**, quay lên cha. | Cây có 6 node; `3` không có con trái; `5` chỉ có con trái `6`. |
| 3 | Đúng khung mục 8. | `tokens.append(...)`; `",".join(tokens)`; `tokens = iter(data.split(","))`. | `node = TreeNode(int(tok)); node.left = build(); node.right = build(); return node`. |
| 4 | Node thật thứ `k` "sở hữu" hai token liên tiếp. | `body = data[1:-1]`; `if not body: return None`. | Trong vòng lặp: đọc trái ở `tokens[i]`, `i += 1`; `if i < len(tokens)` mới đọc phải; `i += 1`. |
| 5 | Mỗi token **lấp** một chỗ trống; node thật **mở** thêm hai chỗ. | `slots` không bao giờ được âm giữa chừng, và phải về `0` ở cuối. | `if slots == 0: return False; slots -= 1; if tok != "#": slots += 2`. |
| 6 | Trong pre-order của BST, cây con trái của `v` là đoạn các giá trị `< v` liền sau `v`. | Cần **nhìn trộm** token kế tiếp mà chưa lấy: dùng list đảo ngược và xem `vals[-1]`. | `if not vals or not (lo < vals[-1] < hi): return None; v = vals.pop(); ...` |

## 19. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
5,3,#,4,#,#,8,#,#
[5,3,8,null,4,null,null,null,null]
[5,3,8,null,4]
5
```

Cây:

```text
      5
     / \
    3    8
     \
      4
```

- DFS: `5`, rồi cây con trái gốc `3` là `3,#,4,#,#` (trái của 3 trống, phải là lá 4), rồi cây con phải `8,#,#`.
- BFS chưa cắt: tầng 0 `5`; tầng 1 `3, 8`; tầng 2 là 4 ô con của `3` và `8`: `null, 4, null, null`; tầng 3 là 2 ô con của `4`: `null, null`. Tổng `9 = 2·4 + 1` token. Cắt 4 `null` cuối còn `[5,3,8,null,4]`.
- Dòng cuối đoán được ngay: cây có `n = 4` node nên có `n + 1 = 5` chỗ trống, tức 5 dấu `#` (mục 5.3).

### Đáp án Bài 2

| Bước | Token | Vị trí đang dựng | Hành động |
|---:|:---:|---|---|
| 1 | `1` | root | tạo node 1 |
| 2 | `2` | trái của 1 | tạo node 2 |
| 3 | `#` | trái của 2 | `None` |
| 4 | `4` | phải của 2 | tạo node 4 |
| 5 | `#` | trái của 4 | `None` |
| 6 | `#` | phải của 4 | `None` → 4 xong → 2 xong |
| 7 | `3` | phải của 1 | tạo node 3 |
| 8 | `#` | trái của 3 | `None` |
| 9 | `5` | phải của 3 | tạo node 5 |
| 10 | `6` | trái của 5 | tạo node 6 |
| 11 | `#` | trái của 6 | `None` |
| 12 | `#` | phải của 6 | `None` → 6 xong |
| 13 | `#` | phải của 5 | `None` → 5 xong → 3 xong → 1 xong |

```text
        1
       / \
      2   3
       \   \
        4   5
           /
          6
```

- Định dạng LeetCode: `[1,2,3,null,4,null,5,null,null,6]`.
- Post-order có `#`: `#,#,#,4,2,#,#,#,6,#,5,3,1`.
- Kiểm tra nhanh: 13 token = `2·6 + 1`, có 7 dấu `#` = `6 + 1`.

Cell kế tiếp kiểm chứng Bài 1 và Bài 2 bằng code; cell sau nữa là **đáp án Bài 3–6** kèm assert.

In [37]:
# Kiểm chứng Bài 1
t = build_tree([5, 3, 8, None, 4])
print(dfs_serialize(t))
print(bfs_serialize(t, trim=False))
print(bfs_serialize(t))
print(dfs_serialize(t).count("#"))

# Kiểm chứng Bài 2
print()
t = trace_dfs_deserialize("1,2,#,4,#,#,3,#,5,6,#,#,#")
print_tree(t)
print("LeetCode  :", bfs_serialize(t))
print("post-order:", post_serialize(t))
assert bfs_serialize(t) == "[1,2,3,null,4,null,5,null,null,6]"
assert post_serialize(t) == "#,#,#,4,2,#,#,#,6,#,5,3,1"

5,3,#,4,#,#,8,#,#
[5,3,8,null,4,null,null,null,null]
[5,3,8,null,4]
5

 1. đọc '1' -> tạo node 1 (root)
 2.     đọc '2' -> tạo node 2 (trái của 1)
 3.         đọc '#' -> trái của 2 = None
 4.         đọc '4' -> tạo node 4 (phải của 2)
 5.             đọc '#' -> trái của 4 = None
 6.             đọc '#' -> phải của 4 = None
            xong node 4 -> trả lên cho cha
        xong node 2 -> trả lên cho cha
 7.     đọc '3' -> tạo node 3 (phải của 1)
 8.         đọc '#' -> trái của 3 = None
 9.         đọc '5' -> tạo node 5 (phải của 3)
10.             đọc '6' -> tạo node 6 (trái của 5)
11.                 đọc '#' -> trái của 6 = None
12.                 đọc '#' -> phải của 6 = None
                xong node 6 -> trả lên cho cha
13.             đọc '#' -> phải của 5 = None
            xong node 5 -> trả lên cho cha
        xong node 3 -> trả lên cho cha
    xong node 1 -> trả về làm root
token còn thừa: []
1
├── L: 2
│   ├── L: ·
│   └── R: 4
└── R: 3
    ├── L: ·
    └── R: 5
        ├── L

In [38]:
# ---------- Bài 3
class AnswerCodecExercise:

    def serialize(self, root):
        tokens = []                                  # TODO 1

        def walk(node):                              # TODO 2
            if node is None:
                tokens.append("#")
                return
            tokens.append(str(node.val))
            walk(node.left)
            walk(node.right)

        walk(root)                                   # TODO 3
        return ",".join(tokens)

    def deserialize(self, data):
        tokens = iter(data.split(","))               # TODO 4

        def build():                                 # TODO 5
            tok = next(tokens)
            if tok == "#":
                return None
            node = TreeNode(int(tok))
            node.left = build()
            node.right = build()
            return node

        return build()                               # TODO 6


# ---------- Bài 4
def answer_bfs_deserialize_exercise(data):
    body = data.strip()[1:-1]
    if not body:
        return None
    tokens = body.split(",")
    root = TreeNode(int(tokens[0]))
    queue, i = deque([root]), 1
    while queue and i < len(tokens):
        node = queue.popleft()
        if tokens[i] != "null":
            node.left = TreeNode(int(tokens[i]))
            queue.append(node.left)
        i += 1
        if i < len(tokens) and tokens[i] != "null":
            node.right = TreeNode(int(tokens[i]))
            queue.append(node.right)
        i += 1
    return root


# ---------- Bài 5
class AnswerSolution331:
    def isValidSerialization(self, preorder: str) -> bool:
        slots = 1                        # ban đầu có 1 chỗ trống: chỗ của root
        for tok in preorder.split(","):
            if slots == 0:               # còn token mà không còn chỗ để đặt
                return False
            slots -= 1                   # token này lấp một chỗ trống
            if tok != "#":
                slots += 2               # node thật mở thêm 2 chỗ cho con
        return slots == 0                # mọi chỗ trống phải được lấp hết


def brute_valid(preorder):
    # Kiểm tra "trâu" để đối chiếu: thử đọc như deserialize, không được thiếu hay thừa token.
    tokens = iter(preorder.split(","))

    def build():
        tok = next(tokens)               # StopIteration nếu thiếu token
        if tok != "#":
            build()
            build()

    try:
        build()
    except StopIteration:
        return False
    return next(tokens, None) is None    # next(it, None): hết thì trả None thay vì báo lỗi


# ---------- Bài 6
def answer_serialize_bst(root):
    return ",".join(str(v) for v in preorder(root))


def answer_deserialize_bst(data):
    if data == "":
        return None
    vals = [int(x) for x in data.split(",")][::-1]   # đảo ngược: token kế tiếp là vals[-1]

    def build(lo, hi):
        if not vals or not (lo < vals[-1] < hi):     # nhìn trộm, chưa lấy
            return None
        v = vals.pop()
        node = TreeNode(v)
        node.left = build(lo, v)
        node.right = build(v, hi)
        return node

    return build(float("-inf"), float("inf"))


def bst_insert(root, v):
    node = TreeNode(v)
    if root is None:
        return node
    cur = root
    while True:
        if v < cur.val:
            if cur.left is None:
                cur.left = node
                return root
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = node
                return root
            cur = cur.right


# ---------- Test
for vals in tests:
    assert run_codec_leetcode(AnswerCodecExercise, vals) == vals
    assert tree_to_list(answer_bfs_deserialize_exercise(bfs_serialize(build_tree(vals)))) == vals
assert AnswerCodecExercise().serialize(build_tree([1, 2])) != AnswerCodecExercise().serialize(build_tree([1, None, 2]))
assert answer_bfs_deserialize_exercise("[]") is None

s331 = AnswerSolution331()
assert s331.isValidSerialization("9,3,4,#,#,1,#,#,2,#,6,#,#") == True
assert s331.isValidSerialization("1,#") == False
assert s331.isValidSerialization("9,#,#,1") == False
assert s331.isValidSerialization("#") == True
checked = 0
for length in range(1, 12):                          # vét cạn mọi chuỗi gồm "1" và "#", tối đa 11 token
    for mask in range(1 << length):
        s = ",".join("1" if mask >> k & 1 else "#" for k in range(length))
        assert s331.isValidSerialization(s) == brute_valid(s), s
        checked += 1
random.seed(331)
for _ in range(200):
    s = dfs_serialize(random_tree(random.randint(0, 25), 0, 100))
    assert s331.isValidSerialization(s)                              # chuỗi thật luôn hợp lệ
    assert not s331.isValidSerialization(s + ",#")                   # thừa token
    if s != "#":
        assert not s331.isValidSerialization(s.rsplit(",", 1)[0])    # thiếu token cuối
print("Bài 5: khớp brute force trên", checked, "chuỗi vét cạn + 200 chuỗi thật")

bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
assert answer_serialize_bst(bst) == "8,3,1,6,4,7,10,14,13"
assert same_tree(answer_deserialize_bst(answer_serialize_bst(bst)), bst)
assert answer_serialize_bst(None) == "" and answer_deserialize_bst("") is None
random.seed(449)
for _ in range(300):
    root = None
    for v in random.sample(range(0, 10_001), random.randint(0, 40)):
        root = bst_insert(root, v)
    assert same_tree(answer_deserialize_bst(answer_serialize_bst(root)), root)
print("Bài 6: BST", answer_serialize_bst(bst), f"({len(answer_serialize_bst(bst))} ký tự) vs DFS có #:",
      dfs_serialize(bst), f"({len(dfs_serialize(bst))} ký tự)")
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Bài 5: khớp brute force trên 4094 chuỗi vét cạn + 200 chuỗi thật
Bài 6: BST 8,3,1,6,4,7,10,14,13 (20 ký tự) vs DFS có #: 8,3,1,#,#,6,4,#,#,7,#,#,10,#,14,13,#,#,# (40 ký tự)
Đáp án tham khảo chạy PASS toàn bộ test!


## 20. Template cuối cùng nên nhớ

**DFS pre-order có `#` (cách chính)** — `O(n)` time, `O(h)` bộ nhớ phụ:

```python
class Codec:

    def serialize(self, root):
        tokens = []

        def walk(node):
            if node is None:
                tokens.append("#")          # chỗ trống cũng phải ghi
                return
            tokens.append(str(node.val))    # node trước
            walk(node.left)                 # rồi trái
            walk(node.right)                # rồi phải

        walk(root)
        return ",".join(tokens)

    def deserialize(self, data):
        tokens = iter(data.split(","))      # một băng chuyền dùng chung

        def build():
            tok = next(tokens)
            if tok == "#":
                return None
            node = TreeNode(int(tok))       # bài 297: TreeNode(x) một tham số
            node.left = build()
            node.right = build()
            return node

        return build()
```

**Câu thần chú**: *"Ghi node, ghi trái, ghi phải — chỗ trống ghi `#`. Đọc lại đúng nhịp đó bằng **một** iterator dùng chung."*

**Khi cần định dạng LeetCode hoặc cây rất sâu**: BFS với queue — ghi cả con `None`, cắt `null` cuối; đọc lại: node thật thứ `k` lấy hai token liên tiếp, hết token thì phần còn lại là `None`.

## 21. Checklist tự kiểm tra trước khi submit

- Chuỗi có ký hiệu cho **mọi** `None` chưa? Có **dấu phân cách** giữa các token chưa?
- Cây rỗng: `serialize(None)` ra gì, và deserialize chuỗi đó có trả `None` không? (Nhớ `"".split(",") == ['']`.)
- Mọi lời gọi đệ quy có dùng **chung một** iterator / chỉ số không (không truyền bản sao list, không dùng `pop(0)`)?
- Kiểm tra `tok == "#"` / `is None`, **không** dùng kiểu truthy làm mất giá trị `0`?
- Số âm, số nhiều chữ số: tách theo token rồi `int(tok)`, không duyệt từng ký tự?
- `TreeNode(int(tok))` đúng **một** tham số, rồi mới gán `.left` / `.right`?
- Không lưu gì trong `self` giữa `serialize` và `deserialize` (hai object khác nhau)?
- Cây lệch sâu `10^4` tầng: đã dùng BFS / stack, hoặc tăng `sys.setrecursionlimit` chưa?
- Đã test round-trip bằng `same_tree`: Example 1, `[]`, 1 node, giá trị `0`, số âm, giá trị trùng, lệch trái, lệch phải, cây ngẫu nhiên chưa?